# IFEST DAC 2026 — V6C Article-Grounded Counterfactual Slot Challenger Ranker

**Frozen champion:** V5 (V2 train-reference graph → V4 test-test canonical matching → V5 positive rescue).  
**V6C intervention:** a component-safe mDeBERTa challenger ranker that asks a *relative* factual question: for a factual slot, does the article support the original title value or an article-grounded alternative more strongly?

This notebook intentionally changes only the residual verification formulation agreed in the V6C design:

- all V2/V4 structural decisions are immutable;
- all V5 positive rescues are immutable;
- no fuzzy graph, no new positive rescue, no synthetic counterfactual generation;
- all 1-edit real same-content canonical↔corrupted train pairs are used, including replace/delete/insert polarity pairs;
- candidate retrieval/evidence is shared across all values within a slot;
- training uses candidate BCE + pairwise margin + positive controls; **no row-level negative MIL**;
- validation first measures challenger-generator coverage, then ranking, then cross-fitted type-specific high-precision vetoes;
- final test changes are only high-confidence `1 → 0` residual-fallback vetoes with 2/3 fold-model same-slot/same-challenger consensus;
- if any precommitted acceptance gate fails, the notebook **fails closed to V5**.

The notebook is designed for Kaggle **T4×2** and does not require any local training.

In [1]:
# Dependency bootstrap + reproducibility configuration.
import os, sys, re, io, gc, json, math, time, gzip, base64, hashlib, random, zipfile, unicodedata, warnings, subprocess, importlib.util, difflib
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext

missing=[]
for pkg,pip_name in [("transformers","transformers>=4.44,<5"),("sentencepiece","sentencepiece"),("sklearn","scikit-learn")]:
    if importlib.util.find_spec(pkg) is None: missing.append(pip_name)
if missing:
    print("Installing missing dependencies:",missing)
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.metrics import f1_score, average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import transformers

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY","1")
pd.set_option("display.max_colwidth",180)

# ---------------- V6C FROZEN CONFIG ----------------
SEED=2026
N_FOLDS=3
N_PSEUDO_SPLITS=100

# Frozen V2/V4/V5 package constants. DO NOT RETUNE HERE.
V2_TITLE_GATE=0.9839511586057019
MIN_GRAPH_CONTENT_WORDS=20
RANK_V3_WEIGHT=0.45
RANK_V2_GLOBAL_WEIGHT=0.55
CONTENT_CANONICAL_MIN_SCORE=0.20
TITLE_CANONICAL_MIN_SCORE=0.90
TITLE_CANONICAL_MIN_MARGIN=0.02
V5_BLEND_MIN=0.67
V5_GLOBAL_ENTAIL_MIN=0.50

# V6C ranker / candidate-generation contract.
MODEL_NAME="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
MODEL_REVISION="b5113eb"
LOCAL_MODEL_PATH=os.environ.get("V6C_MODEL_PATH","").strip() or None
MAX_LENGTH=384
SLOT_WINDOW_WORDS=80
SLOT_WINDOW_STRIDE=40
SLOT_TOP_WINDOWS=3
MAX_CHALLENGERS=8
MAX_LEXICAL_SLOTS=2
MAX_NUMERIC_SLOTS=3
MAX_POLARITY_SLOTS=3
HIGH_IDF_QUANTILE=0.80
PAIR_SIMILARITY_MIN=0.80
MAX_CHANGED_SPAN_TOKENS=3
PAIR_MARGIN=0.50
CANDIDATE_BCE_WEIGHT=1.0
PAIR_LOSS_WEIGHT=0.50
POSITIVE_CONTROL_WEIGHT=0.25

# Legacy V2 evidence defaults are retained only because frozen graph helpers define global_evidence().
GLOBAL_WINDOW_WORDS=90
GLOBAL_WINDOW_STRIDE=45
GLOBAL_TOP_WINDOWS=3

# T4x2 budget.
EPOCHS=2
CONTROL_BATCH_SIZE=4
EVAL_SEQUENCE_BATCH=48
GRAD_ACCUM=4
LR=1.0e-5
WEIGHT_DECAY=0.01
WARMUP_RATIO=0.08
MAX_GRAD_NORM=1.0
USE_AMP=True
NUM_WORKERS=2

# Precommitted V6C acceptance contract.
MIN_GENERATOR_RECALL=0.90
MIN_LEXICAL_GENERATOR_RECALL=0.90
MIN_NUMBER_GENERATOR_RECALL=0.85
MIN_POLARITY_AVAILABILITY=0.95
MIN_TOP1_RANKING=0.90
MIN_COVERAGE_ADJUSTED_PAIR_SUCCESS=0.80
MIN_VETO_PRECISION=0.90
MIN_FOLD_PRECISION=0.80
MIN_CORRECTED_OOF_NEGATIVES=10
MIN_PSEUDO_WINS=80
MAX_PSEUDO_LOSSES=5
MIN_TEST_VETOES=2
MAX_TEST_VETOES=30
MIN_THRESHOLD_SELECTED=2

EXPECTED_TRAIN_ROWS=14397
EXPECTED_TEST_ROWS=3603
EXPECTED_V5_CHANGES_VS_V4=4
OUTPUT_DIR=Path("/kaggle/working" if Path("/kaggle/working").exists() else "/mnt/data/ifest_v6c_output")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_COUNT=torch.cuda.device_count()
AMP_ENABLED=bool(USE_AMP and DEVICE.type=="cuda")
if DEVICE.type!="cuda":
    raise RuntimeError("V6C training is designed for Kaggle GPU. Select GPU T4 x2 before running.")
if GPU_COUNT>=2: print("T4x2 path enabled via torch.nn.DataParallel")
else: print("WARNING: fewer than 2 CUDA devices found; notebook will use the available GPU without changing methodology.")
for i in range(GPU_COUNT):
    p=torch.cuda.get_device_properties(i); print(i,torch.cuda.get_device_name(i),f"VRAM={p.total_memory/1024**3:.1f} GB")
print(f"PyTorch={torch.__version__} | Transformers={transformers.__version__} | AMP={AMP_ENABLED}")

T4x2 path enabled via torch.nn.DataParallel
0 Tesla T4 VRAM=14.6 GB
1 Tesla T4 VRAM=14.6 GB
PyTorch=2.10.0+cu128 | Transformers=5.0.0 | AMP=True


In [2]:
# Frozen prior-experiment artifacts embedded for a self-contained Kaggle handoff.
# These are predictions from already-run V2/V3/V5; no hidden-test labels are embedded.
FROZEN_ARTIFACTS={"V2_OOF": {"sha256": "4e5fe37b48033f4fff85ecbbd569df0c991e9435f3c1330f68096dc2dbdb58d8", "rows": 14397, "payload_b64_gzip": "..."}, "V2_TEST": {"sha256": "3d0fc35dc3525f755ac3a45270b5bd474f71090e1d77d9c1c666ba79762cab6e", "rows": 3603, "payload_b64_gzip": "..."}, "V3_OOF": {"sha256": "a57f2ddab5848365ba90dc57da156e8752758fa0231534b118c8179226188a2d", "rows": 14397, "payload_b64_gzip": "..."}, "V3_TEST": {"sha256": "7a7911d937746215ed29d50037ecfdbd84876bb95659fae6b308876a23a39578", "rows": 3603, "payload_b64_gzip": "..."}, "V5_TEST": {"sha256": "56d692997afc57493c5b8a8ca8937f62c2bc27fff30ec252867ee95af24b3ecd", "rows": 3603, "payload_b64_gzip": "..."}, "V5_SUBMISSION": {"sha256": "698790750562fda1dcc7eea78ac428d079312254b535f9aedf85fb240422dfdc", "rows": 3603, "payload_b64_gzip": "H4sIADmSm2oC/z3dS25k63WE0b7HogYj4jyHY8NqGFDT84dFmGeplWBVad+TWV/9+Vjc/J///se//vO//vmv//jff/78/u8fP3+38o/83apbc+tw63Trcut263Hr/W7lxy0zYkbMiBkxI2bEjJgRM2pGzagZNaNm1Iya0fu7N2pGzZgZM2NmzIyZMTNmxlzHzJgZhxmHGYcZhxnH8f2XHmYcl6+ZcTy+ZsZpxmnGacZpxuk6TjNO13GacbqO04zLjMuMq99/y2XGZcZlxuU6LjMuMy4zbjNuM27XcZtxm3GbcbuO24zbjNuMx4zHjMeMx4zHjMeMx4zHjMeM5/2u8jXjja+Z8ZrxmvGa8ZrxmvGa8X7XkZ8ft+LW93hE59F5dB6d5+f2Jx5fM0Pn0Xl0Hp1H59F5dB6dJ9/f3eS7r6Lz6Dw6j86j8+g8Ok/N0Hl0Hp1H59F5dB6dR+fReXQenUfn0Xl0nsPjofMcZug8hxmHx0Pn0Xl0nvP7exWdR+fReXQenUfn0Xl0Hp1H57nM0Hl0Hp3nMkPnuVyHzqPz6Dy3+0rn0Xl0Hp1H59F5dB6dR+fReXSe5/t3NzqPzqPz6DyPGTrPa4bOo/PoPDqPzqPz6Dw6r86r8zrPq/PqvDqvzus878/3eFTn1Xl1Xp1X59V5dV6dV+d1nlfn1Xl1Xp33351/v88MnVfn1Xl1Xp1X59V5dV6dd99jXp1X593tV91XOu/xPb+qzus8r86r8+q8zvM6z6vz6rynGc7z6rw6r86r8+q8Oq/Oq/PqvDqvzqvz6rw6r86r8zrPq/PerkPndZ5X59V5/9359yfM0Hl1Xp1X59V5dV6d13lenffxeDwec51X59V5dd73+/eqOq/Oq/O+Zui8Oq/Op/PpfDqfzvfz/d2dzvdz+drta49bZuh8Op/Op/PFDJ1P59P5dD7P29fvMV/jlhnO8+l8Pf0+Mzxvn86n8+l8Op/Op/M5z6fz6XzO8znPp/M5z6fz6Xw6n86n8znPp/MdZuh8zvPpfDqfzqfz6Xw6n86n8+l8Op/Op/PpfDqfzqfz6Xw63/X92z7n+XQ+ne+e32eG83w6n86n8+l8Op/Op/PpfDqfzuc8n/N8Op/n7dP5dD6dT+fT+XQ+nU/n0/l0fuj80Pmh80Pnh/P80PnhPD90fuj80Pmh80Pnh84PnR/O80Pnh84PnR86P3R+OM8PnR86P5znRw+3TrfMcJ4fOj90fuj8mBk6P3R+6PzQ+aHzQ+eHzg+dHzo/dH543n7o/ND5ofND54fOD50fOj90fuj80Pmh80Pnh84PnR86P3R+6PzQ+aHzQ+eHzg+dHzo/dH7o/PC8/XCeHzo/dH7o/HCeHzo/dH7o/ND5ofND54fOD50fOj90fuj80Pmh80Pnh84P5/mh80Pnh+fth84PnR86PzxvP3R+6PzQ+anzU+en1+enzk+dnzo/neenzk+dnzo/dX7q/NT5qfNT56fOT52fOj+9Pj91fur89Lz91PnpPD+d56fn7afOT52fOj91fur8dJ6fOj91fur81Pmp81Pnp85PnZ86P3V+6vzU+anzU+enzk+dnzo/dX7q/NT5qfNT56fOT52fOj/P77nPqfNT56fOT52fOj91fur8vL5/d0+dnzo/neenzk+dnzo/vT4/dX7q/NT5qfNT56fOT52fOj91fur81Pmp81Pnp85PnZ86P3V+6vzU+anzU+enzs/XfaXzU+enzs/3e85w6fzS+eU8v3R+6fzS+eU8v3R+6fzS+aXzS+dXvsfjytwyQ+eXzq/cfp8ZOr90fun80vml80vnl86vfv9eXTq/dH7p/NL5pfNL55fOL6/PL51fOr90fnl9fs3jofNL55fn7dfhvtL5pfNL55fOL51fOr90fun80vml8+t0HTq/dH45zy+dXzq/dH7p/NL5pfNL55fz/NL5pfNL55fOL51fOr+c55fn7ZfOL51fOr90fun80vml80vnl84vnV86v3R+6fzy+vx6POY6v3R+6fzS+fW6Dp1fzvNL55fOL51fzvNb57fOb53fOr91fuv81vmt81vnt85vnd86v53nt/P81vmt81vnt/P8dp7fOr91fuv81vmt89vz9lvnt/P81vmt81vnt85vnd86v3V+6/zW+a3zW+e38/x2nt86v3V+6/x2nt8+V7t1fuv81vmt81vnt85vnd86v3V+O89vnd86v3V+nx4Pnd86v3V+6/zW+a3zW+e3zm+d3zq/dX7r/L6/11G3zm/n+a3zW+e3zm+d3zq/dX7r/H7M0Pmt81vn93P6fWY4z2/n+e31+a3zW+e3zm/n+a3z23l+6/zW+a3zW+ePz88fnT86f3T+6PzR+aPzR+ePzh+dPzp/dP7o/NH5o/Mn33U8On90/jjPH50/On90/uj80fnjPH90/uj80fmj80fnj84fnT86f/b9u/vo/Jnr0Pmj80fnj84fnT86f3T+6Pxxnj86f3T+6PzR+aPzR+ePz9UenT86f3T+6PzR+aPzx3n+6PzR+aPzR+ePzh+dPzp/dP5c3zn46PzR+eP99sd5/uj80fmj80fnj84fnT86f3T+6PzR+aPz5/GY6/xxnj/eb390/jjPH+f5o/NH54/OH50/On+c54/OH50/On90/jrPX52/On91/ur81fnr9fmr81fnr85fnb86f3X+6vzV+es8f3X+6vzV+avzV+evzl+dvzp/df7q/NX5q/NX56/OX52/On91/ur89fr8dZ6/On91/ur81fmr81fnr9fnr85fnb86f3X+6vzV+avzV+ev8/zV+avzV+evzl+dvzp/df7q/NX5q/NX56/OX52/On91/ur81fmr89d5/ur81fmr81fnr85fnb86f3X+6vx9vn9LXs/b38fj4Tx/df7q/HWevzp/df7q/NX5y8m8On91/ur81fmr8/d1X+n8/TrPz9d5fj3cz9+t+trcOtw63br8idvXHrfMiBlf5+HhwsOFhwsPFx4uP9/r8/Bw+fk+Pw8PFx4uPFx+vvfbw8OFhwsPl5+6jppR1zEzZsbMmOv4zvPwcOHhwsPl53t9Hh4uPFx4uPBw4eHyc7iOw4zDjMOMw3Uc7qvTjNOM04zTjNOM04zTjNOM03WcruMy4zLjMuPyeFxmXGZcZlxmXGZcZtxm3GbcZtyu4zbjNuM24zbjNuM24zHjMeMx4zHjMeMx4zHjMePxeDwej9eM14zXjNeM19+r14zXjNeM13XonIcLDxceLjxceLjwcInOo3MeLjxceLj8eri/P6FzHi48XHi48HDh4RKdJ2bonIcLDxceLjxceLikrkPnPFx4uPBw4eHCw4WHCw8XHi48XPI5mfBwyfc+XHi4/Hq472tm6JyHCw8XHi7ReXQenfNwic5/Pdz3q2bonIcLDxceLjxceLjwcOHhwsOFhwsPFx4uPFx4uOQyQ+c8XHi4ROf53ocLDxceLjxceLjwcOHhwsOFh0se95XOo3MeLjxceLjwcOHhwsOFhwsPFx4uPFx4uPBwyevv1WuGznm48HDh4cLDhYcLDxceLjxcqnMeLjxceLjwcOHh0s/JhIcLDxceLjxcqnMeLjxceLhU560ZOufhwsOFhwsPFx4uPFx4uPBwqc55uPBwqfO8Oq/Oq3MeLjxceLjwcOHhUp33c6+pznm48HDh4cLD5dfDfb/qvtI5DxceLjxceLjwcOHh0s+3h4cLDxceLjxceLjwcOHhUp33ex8uPFx4uPBw4eHCw4WHS53nPFz6uK90zsOFhwsPlz5m6Lw65+HCw4WHCw+X6pyHCw8XHi7VOQ8XHi48XHi48HDh4cLDZTr/9XA/f7cut26/+rhlhs55uPBw4eHCw4WHCw8XHi48XHi4TOfTOQ8XHi48XKZzHi7T+a+H+/tv6dcgDxceLr8e7vtVMzxv5+HCw4WHCw8XHi48XHi47DBD5zxceLjwcOHhssN16JyHCw8XHi48XHi48HDh4cLDhYcLDxceLjxceLjwcPn1cH//fTrn4cLDhYfLdM7DhYfLbveVznm48HDh4cLDhYcLD5fpnIcLDxceLjxceLjwcOHhwsNlOufhwsNlOufhMs/bebhM5zxceLjwcOHhwsOFhwsPFx4uPFx4uPBwOT7fHh4uvx7u72uetx/f5+fh4cLD5fjebw8PFx4uPFx4uPBw4eHCw+XQ+aHzQ+c8XHi48HDh4cLD5dA5DxceLjxceLjwcOHhwsOFh8txuK90fjjPebjwcOHhwsOFh8uhcx4uPFx4uPBw4eHCw4WHCw8XHi48XHi4/Hq4v7k65+FyeH3Ow4WHCw8XHi6Hzo/Pvea4zdA5DxceLjxceLjwcOHhwsOFhwsPFx4uPFx4uBzf97GEh8vxvd8eHi48XHi48HDh4cLDhYcLD5fj1YfOebjwcOHhwsOFh8upcx4uPFx4uJw6P7/vVwsPFx4uPFx4uPBw4eHCw4WHCw8XHi6nznm48HDh4cLDhYfL6Xk7DxceLjxceLjwcOHhwsOFhwsPFx4uPFx4uPBw4eHCw+XUOQ+XU+c8XHi48HDh4cLDhYcLDxceLjxceLjwcOHhwsOFh8upcx4uPFx4uPBw4eHCw4WHy6lzHi48XHi48HDh4cLD5dQ5DxceLjxceLjwcOHhwsOFhwsPFx4uPFx4uPBw4eHCw4WHCw8XHi48XHi48HDh4XI6z3m4nF6f83A5dc7DhYcLDxceLjxceLjwcOHhcjnPebjwcOHhcun80jkPFx4uPFwunfNw4eHCw4WHCw8XHi48XHi48HDh4cLDhYcLDxceLjxceLhcOufhwsOFh8ul80vnPFx4uPBw4eFyfU4mPFx4uPBw4eHCw4WHy/V9fh4eLjxcLp3zcOHhwsOFhwsPFx4uPFx4uPBw4eHCw4WHCw8XHi48XHi48HDh4cLDhYcLDxceLjxceLjwcOHhwsOFhwsPFx4ul9fnPFwunV86v7zfzsOFhwsPl0vnPFx4uFyvv7vO88v77TxceLjwcOHhwsOFhwsPFx4uPFx4uPBw4eHCw4WHCw8XHi48XH493M/fLTN0zsPl/r5fLTxceLjwcLl1zsOFhwsPFx4uPFx4uPBw4eHCw+XWOQ8XHi48XHi48HDh4cLD5fb6nIfL7Tzn4cLDhYcLDxceLjxcbp3zcOHhwsOFhwsPFx4ut855uPBw4eHCw4WHCw8XHi48XHi48HDh4cLDhYcLDxceLjxceLj8erjvT5ih818P9/erOr91zsOFhwsPl9t5zsOFhwsPl9t5zsOFhwsPFx4uPFxunfNw4eHCw4WHy6NzHi48XHi48HDh4cLDhYfL4zzn4cLDhYcLDxceLjxcHp3zcOHhwsPl8fk5DxceLjxceLjwcHm+71fL4zzn4cLDhYcLDxceLjxcHp0/On90zsOFh8vj83MeLjxceLg8OufhwsOFhwsPFx4uPFx4uDzeb388b398rsbDhYcLDxceLjxceLjwcOHhwsOFh8vj9fnjfTgeLjxcHp3zcOHh8uichwsPFx4uPFx4uPBw+fVwf/8FOufhwsPl0TkPl0fnPFx4uDw6fx4zdM7DhYcLDxceLjxceLjwcOHh8njezsOFhwsPFx4uPFx4uPBw4eHy6pyHCw8XHi48XHi48HDh4cLDhYcLDxceLjxceLjwcOHhwsOFhwsPFx4uPFx4uPBw4eHy6+F+/m6ZoXMeLjxceLj8erjvV83QOQ8XHi48XF7P23m48HDh4cLDhYcLDxceLjxceLjwcOHhwsOFhwsPFx4uPFx+PdzfFemchwsPFx4uPFx4uPBweXXOw4WHCw8XHi48XHi48HDh4cLDhYcLDxceLq/O38+95tU5DxceLjxceLi8On91zsOFh8urcx4uPFx4uLzeh+Ph8jrPebjwcOXhaj9cebjycOXhysPVfrjycOXhysOVhysP15/v/fbycOXhysOVh6v9cP35vo+l9sOVhysPVx6u9sOVh+vPd56XhysP15+aUTNmxsyYGTNjZsx1zIyZMffVzDjM+JxMebjycOXhysOVhysPV/vhaj9cebjycOXhysP15/v8vDxcebjycLUfrjxcf7732/vr4b5bZlxmXK7jMuMy4zLjMuN7v708XHm48nDl4crDlYcrD1cervbDlYcrD1cerjxcfz7fXh6uPFx5uPJw5eH685jxmvGa8Zrxuo7XjNeM14zXjNcMnfNw5eHKw5WHKw9XHq72w5WHKw/XXw/39/+ic/vhysOVhysPVx6uPFx5uPJw5eHKwzXf52rl4crDlYcrD1f74crDlYdr6jp0zsOVhysPVx6uPFx5uNoPVx6u9sM13/vttR+uPFx5uPJw5eHKw5WHq/1wtR+uPFx5uNoPVx6uPFx5uPJw5eHKw5WHKw9XHq48XHm48nDl4crDlYcrD1cerjxcebjycLUfrjxc8zmZ8nDl4crDlYdrHo+Hzu2HKw9XHq55zNB5vvO8PFx5uPJw5eHKw5WHKw9XHq48XO2HKw9XHq79vl+tPFx5uPJw5eHKw5WHKw9XHq48XHm48nDl4crDlYcrD1cerjxcebjycOXhysOVhysPV/vhysOVhysPVx6uPFx5uPJw5eHKw5WHa7/vVysPVx6uPFzthysPVx6uPFx5uPJw/fVwfzO+/XDl4crDlYcrD9d+78OVh6v9cLUfrjxcebjycOXhysOVh2ud5/bDlYdrL4+5znm48nDl4crDlYer/XDl4crDlYdrnec8XHm42g9XHq72w5WHKw9XHq48XO2HKw9XHq48XHm48nDl4crDlYcrD1cerjxcebjycOXhysOVhysPVx6uPFzth+t0zsOVhysP131OpvucTHm48nCd85yHKw9XHq48XHm48nDl4crDlYcrD1cerjxc7Yer/XC1H648XHm42g9XHq72w5WHKw9XHq48XHm48nDl4Wo/XHm48nDl4crDlYer/XDd59vLw5WHKw9XHq48XHm4/nq47//FDJ3zcOXhysOVhysPV/vhysOVhysPVx6u9sOVhysPV/vhysOVhysPVx6uPFx5uPJwtR+ue/zd1TkPVx6uPFx5uM55zsOVhysPVx6uPFx5uNoPVx6uPFx5uPJw5eHKw5WHKw9XHq48XHm48nA9PidT++HKw9V+uNoPV/vhysOVh6v9cOXhysOVhysPVx6uh855uPJw5eHKw5WHKw9XHq48XHm48nDl4crDlYcrD1cerjxcebjycLUfrjxcebjycOXhysOVhysPV/vhysOVhysPVx6uPFx5uPJw5eHKw5WHKw9X++HKw5WHq/1w5eHKw5WHKw9XHq72w9V+uPJw5eHKw5WHKw9XHq48XHm48nDl4crDlYcrD1f74crD1X648nDl4Xp8n5+XhysPVx6uPFx5uNoPVx6uPFx5uB7f3qjycOXhysOVhysPVx6u9sPVfrjycD29PufhysOVh+v57Y0qD1cerjxcebjycOXhysOVhysPVx6u9sOVhysPVx6uPFx5uPJw5eHKw5WHKw9XHq48XM/v+9XKw5WHq/1w5eHKw5WHKw9XHq48XHm48nDl4crDlYcrD1cerjxcebjycOXhen7fl1oerjxcebjycOXhysOVh+t5+Xul8/MyQ+c8XHm48nDl4frr4f7+rM55uPJw5eHKw5WHKw/X8/t+tZ7eh+PhysOVhysPVx6uPFx5uPJw5eHKw/V8XYfOebjaD1cervbDlYcrD1cerr8e7vt9h6+dbl1u3W49/sR3HTxcL8/bebjycOXhysOVhysPVx6uPFx5uPJw5eHKw5WHKw/XS+c8XC/vt/Nw5eHKw/X6vo+lPFx5uPJwvb7P1crDlYcrD1cerjxcebheOrcfrjxcebjycOXhennezsPVfrhep+vQuf1w5eHKw/X69kCWhysPVx6uPFx5uPJw5eHKw5WHKw9XHq48XHm48nDl4Xrp/NI5D9fr9njonIcrD1cerjxcebjycOXhaj9cebjycOXhaj9cebjycOXhysOVhysPV/vhysPVfrjycL29D8fDlYcrD1cerjxcebjycOXhysOVhysPVx6uPFx5uN7f97HUfrjycOXhysPVfrjycOXhysOVhysPVx6uPFx5uPJw5eHKw5WHKw9X++HKw/XWOQ/X+/v8vDxcb++383C1H648XHm48nC9vQ/Hw5WHKw9XHq72w5WHKw9XHq48XHm48nDl4crDlYcrD1cerjxcebjycOXhysOVhysPVx6ut855uPJw5eHKw9V+uPJwvW+Ph855uPJw5eF6O8/vx2Oucx6ut9fnPFx5uP56uL9f1TkP1/s1Q+c8XHm48nDl4crD1X648nDl4fo4z5/v5zSUhysPVx6u9sOVhysP18d5zsOVhysPV/vhysOVhysPV/vhysOVhysPVx6uPFzth6v9cOXhysOVh+sz95XOebjycOXhysOVh+vjebv9cOXhysOVh6v9cOXhysOVh+ujcx6uPFx5uNoPVx6u9sOVhysPVx6uPFx5uPJw5eHKw5WHKw9XHq72w5WHKw9XHq48XHm48nDl4crDlYfr4zzn4crD1X648nB9PvdaHq48XHm48nDl4crDlYer/XDl4crDlYcrD1cerjxcH8/bebjycLUfrjxcebjycOXhysOVhysPVx6uPFx5uPJw5eHKw5WH68vJ8HDl4crDlYcrD1cerjxcebjycOXh+n57o8rDlYcrD1cerq/n7Txc7YcrD1cerq/OX+c5D1f74crD1X648nDl4crDlYcrD1cerjxcX53zcOXhysOVhysPVx6uPFx5uPJw5eHKw5WHq/1w5eH6+lyNhysPVx6uPFx5uPJw5eHKw5WHKw9XHq48XF+d83Dl4crDlYfrq3MerjxcebjycOXhysOVh6v9cOXhysOVhysPVx6uPFx5uNoPVx6uPFx5uPJw4+HGw42HGw83Hm483Hi48XDj4cbDjYcbDzcebjzceLjxcOPhxsPNfrjxcOPhxsONhxsPNx5u9sONhxsPN/vhxsONhxsPNx5uPNx4uPFw4+HGw42HGw83Hm72w+3ncF8dZnzfrzYebn5e6ni42Q83Hm483Hi48XDj4WY/3Hi48XDj4cbDjYeb/XCzH2483Hi48XDj4cbD7efbAzkebvbDjYcbDzcebjzceLjxcOPhxsONhxsPNx5uPNx4uNkPNx5uPNx4uPFw4+FmP9x+vuft4+HGw42HGw83Hm483Hi48XDj4cbDjYdbdM7DjYcbDzcebjzceLjZDzcebjzceLjle799+ZzMeLjxcOPhxsONhxsPN/vhxsONhxsPNx5uPNzyfX4+Hm72wy3f8/bxcOPhxsMtcx06z/f6fDzceLjxcLMfbjzceLjxcOPhxsONh5uflzr74WY/3Hi48XDj4cbDjYcbD7ecrkPn+d5vHw83Hm75zvPl2xs1Hm483Hi48XDj4cbDjYcbDzcebjzceLjxcLMfbjzceLjxcOPhZj/ceLjxcOPhxsONhxsPN/vhlsd99ZjxmqFzHm483Hi48XDj4cbDLa8Z3/vt4+HGw42HGw83Hm799kaNh1u/z8/Hw42HGw83Hm483Hi4+Xmp4+HGw60xQ+d+Xup4uPFw4+HGw42HGw83Hm483Po9bx8PNx5uPNx4uPFw4+FmP9x4uPFw4+HGw42HGw83Hm483Hi42Q83++Hm56WOh1t1Xp3bDzcebjzceLjxcOPhxsOtOufhxsPNfrjxcOPhZj/ceLjxcOPhxsONhxsPNx5uPNx4uPFw4+HGw42HGw83Hm483Hi48XDj4cbDjYcbDzcebjzceLjxcOPhxsOt3/tw4+HGw42HGw+36pyHGw83Hm483Hi48XDj4cbDjYcbDzcebvbDzX648XDj4cbDjYcbDzcebjzceLj5eanj4cbDjYcbDzcebvu+L3X2w42HGw83Hm77fo7ieLjxcOPhxsNtznMebjzceLjxcOPhxsONhxsPNx5uPNx4uPFwsx9uPNx4uPFw4+HGw42Hm/1w4+HGw23f96uNhxsPNx5uPNx4uPFw4+HGw42Hm/1w4+HGw42HGw83Hm483Hi48XDj4cbDjYeb/XDj4bbv/fbte799c57zcOPhxsONhxsPNx5uPNx4uPFw4+HGw42H23TOw42HGw83Hm7H9377eLjxcOPhxsONhxsPNx5uPNx4uPFw4+HGw42HGw83Hm483Hi48XDj4ebnpY6HGw83Hm483Hi48XDj4cbD7fg+Px8PNx5uPNx4uPFw4+HGw42HGw83Hm483A6vz3m48XDj4cbD7XCe83Dj4cbDjYcbDzcebjzceLjxcOPhxsONh9vheTsPNx5uPNx4uPFw4+HGw42HGw+34zZD5zzceLjxcOPhxsONhxsPNx5uPNx4uPFw4+HGw42HGw83++HGw42HGw83Hm483Hi4Hd8+mfFwO3RuP9x4uPFw4+HGw42H2+l5u/1w4+F2fk5m9sONhxsPNx5u9sONhxsPNx5uPNx4uPFw4+HGw42HGw83Hm483Hi48XDj4cbDjYcbDzcebjzceLjxcOPhZj/ceLjxcLMfbjzceLjxcOPhxsPt9D4cDzcebjzceLjxcOPhxsONhxsPNx5uPNzshxsPNx5uPNx4uPFw4+Hm56XOfrjxcLMfbjzceLjxcOPhxsPNfrjxcOPhxsONhxsPNx5uPNzsh5v9cOPhdjrPebidznMebjzceLjxcOPhxsPNfrjxcDt1zsONh5v9cOPhxsPt/z3c/wGx/IAwx4wAAA=="}}

def decode_frozen_csv(spec):
    raw=gzip.decompress(base64.b64decode(spec["payload_b64_gzip"]))
    got=hashlib.sha256(raw).hexdigest()
    if got!=spec["sha256"]: raise RuntimeError(f"Frozen artifact checksum mismatch: {got} != {spec['sha256']}")
    df=pd.read_csv(io.BytesIO(raw))
    if len(df)!=int(spec["rows"]): raise RuntimeError("Frozen artifact row-count mismatch")
    return df

v2_oof=decode_frozen_csv(FROZEN_ARTIFACTS["V2_OOF"])
v2_test=decode_frozen_csv(FROZEN_ARTIFACTS["V2_TEST"])
v3_oof=decode_frozen_csv(FROZEN_ARTIFACTS["V3_OOF"])
v3_test=decode_frozen_csv(FROZEN_ARTIFACTS["V3_TEST"])
v5_test=decode_frozen_csv(FROZEN_ARTIFACTS["V5_TEST"])
v5_submission=decode_frozen_csv(FROZEN_ARTIFACTS["V5_SUBMISSION"])
print("Frozen artifacts verified:",{k:(v["rows"],v["sha256"][:12]) for k,v in FROZEN_ARTIFACTS.items()})

Frozen artifacts verified: {'V2_OOF': (14397, '4e5fe37b4803'), 'V2_TEST': (3603, '3d0fc35dc352'), 'V3_OOF': (14397, 'a57f2ddab584'), 'V3_TEST': (3603, '7a7911d93774'), 'V5_TEST': (3603, '56d692997afc'), 'V5_SUBMISSION': (3603, '698790750562')}


In [3]:
def find_competition_csv(filename):
    roots = [Path("/kaggle/input"), Path("/mnt/data/ifest_dac")]
    candidates = []
    for root in roots:
        if not root.exists(): continue
        candidates.extend(root.rglob(filename))
    # Avoid output/working mirrors; prefer the smallest path depth and a parent that contains all three files.
    candidates = [p for p in candidates if "/kaggle/working/" not in str(p)]
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input. Attach the competition dataset.")
    candidates = sorted(candidates, key=lambda p:(len(p.parts), len(str(p))))
    return candidates[0]

train_path = find_competition_csv("train.csv")
test_path = find_competition_csv("test.csv")
sample_path = find_competition_csv("sample_submission.csv")
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)

assert list(train.columns) == ["id","title","content","label"], train.columns.tolist()
assert list(test.columns) == ["id","title","content"], test.columns.tolist()
assert set(sample.columns) >= {"id","label"}
assert len(train) == EXPECTED_TRAIN_ROWS and len(test) == EXPECTED_TEST_ROWS
assert train.label.isin([0,1]).all()
assert train.id.astype(str).tolist() == v2_oof.id.astype(str).tolist() == v3_oof.id.astype(str).tolist()
assert test.id.astype(str).tolist() == v2_test.id.astype(str).tolist() == v3_test.id.astype(str).tolist() == v5_test.id.astype(str).tolist()
assert test.id.astype(str).tolist() == v5_submission.id.astype(str).tolist() == sample.id.astype(str).tolist()
assert np.array_equal(train.label.to_numpy(dtype=np.int8), v2_oof.label.to_numpy(dtype=np.int8))
assert np.array_equal(train.label.to_numpy(dtype=np.int8), v3_oof.label.to_numpy(dtype=np.int8))
assert np.array_equal(v5_test.label.to_numpy(dtype=np.int8), v5_submission.label.to_numpy(dtype=np.int8))
if {"v4_label","v5_rescue","label"}.issubset(v5_test.columns):
    assert int((v5_test.v4_label.to_numpy(np.int8)!=v5_test.label.to_numpy(np.int8)).sum())==EXPECTED_V5_CHANGES_VS_V4
    assert int(v5_test.v5_rescue.to_numpy(np.int8).sum())==EXPECTED_V5_CHANGES_VS_V4


def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20), b""): h.update(chunk)
    return h.hexdigest()

data_hashes = {"train":sha256_file(train_path),"test":sha256_file(test_path),"sample_submission":sha256_file(sample_path)}
print("Dataset:", train_path.parent)
print("Shapes:", train.shape, test.shape, sample.shape)
print("Dataset SHA256:", data_hashes)

Dataset: /kaggle/input/competitions/penyisihan-ifest-2026-dac
Shapes: (14397, 4) (3603, 3) (3603, 2)
Dataset SHA256: {'train': 'a4ba91606302455da8a9523906a25361d581ac15b108964cdb3ab6beb0adb5d8', 'test': '8e5de5bf53a50df87270dccdaba5ed0700d3729b5b03e4ce7724ac1769177005', 'sample_submission': 'ea09389db317450c7b93da76d87b771daca79e66716b84b7432f1653a67582c6'}


## Frozen V2→V4→V5 champion reconstruction

The following helpers are copied from the frozen prior stack so V6C changes no incumbent graph, calibration, or rescue behavior.

In [4]:
TOKEN_RE = re.compile(r"[\w]+", flags=re.UNICODE)
SOURCE_SUFFIX_RE = re.compile(r"\s+kumparan\s+com\s*$", re.I)


def clean_ws(text):
    text = unicodedata.normalize("NFKC", str(text)).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()

def clean_title_for_model(text):
    return SOURCE_SUFFIX_RE.sub("", clean_ws(text)).strip()

def normalize_key(text):
    return " ".join(TOKEN_RE.findall(clean_ws(text).lower()))

def tokens(text):
    return TOKEN_RE.findall(normalize_key(text))

for df in (train, test):
    df["title_model"] = df.title.map(clean_title_for_model)
    df["title_key"] = df.title_model.map(normalize_key)
    df["content_key"] = df.content.map(normalize_key)
    df["pair_key"] = df.title_key + "|||" + df.content_key
    df["content_words"] = df.content.map(lambda x: len(clean_ws(x).split()))

# Label-free IDF fit on the supplied train corpus, matching V2 evidence construction.
doc_freq = Counter(); N_DOCS=len(train)
for t,c in tqdm(zip(train.title_model,train.content), total=N_DOCS, desc="IDF corpus"):
    doc_freq.update(set(tokens(t)) | set(tokens(c)))
idf = {tok:math.log((N_DOCS+1)/(df+1))+1.0 for tok,df in doc_freq.items()}
IDF_FLOOR = math.log((N_DOCS+1)/2)+1.0

def tok_idf(tok): return idf.get(tok, IDF_FLOOR)
def title_window_score(title_toks, window_text):
    tset=set(title_toks)
    if not tset: return 0.0
    wset=set(tokens(window_text)); denom=sum(tok_idf(t) for t in tset)+1e-9
    return sum(tok_idf(t) for t in tset if t in wset)/denom

def make_windows(content, window_words=GLOBAL_WINDOW_WORDS, stride=GLOBAL_WINDOW_STRIDE):
    words=clean_ws(content).split()
    if not words: return [(0,"")]
    if len(words)<=window_words: return [(0," ".join(words))]
    starts=list(range(0,max(1,len(words)-window_words+1),stride)); last=max(0,len(words)-window_words)
    if starts[-1]!=last: starts.append(last)
    return [(s," ".join(words[s:s+window_words])) for s in starts]

def global_evidence(title, content):
    tt=tokens(title); windows=make_windows(content)
    scored=sorted([(title_window_score(tt,text),i) for i,(_,text) in enumerate(windows)], reverse=True)
    selected=[0]
    for _,i in scored:
        if i not in selected: selected.append(i)
        if len(selected)>=GLOBAL_TOP_WINDOWS: break
    lead=[i for i in selected if i==0]
    rest=[i for _,i in scored if i in selected and i!=0]
    return " \n ".join(windows[i][1] for i in lead+rest)


def build_graph_reference(ref_df):
    pair_to_label={}
    for key,g in ref_df.groupby("pair_key",sort=False):
        labs=set(map(int,g.label))
        if len(labs)==1: pair_to_label[key]=next(iter(labs))
    content_to_pos_titles=defaultdict(set); title_to_pos_contents=defaultdict(set)
    for r in ref_df.loc[ref_df.label==1].itertuples():
        content_to_pos_titles[r.content_key].add(r.title_key)
        title_to_pos_contents[r.title_key].add(r.content_key)
    return pair_to_label,content_to_pos_titles,title_to_pos_contents


def frozen_v2_graph_decisions(query_df, ref_df, v2_meta_prob):
    q=query_df.reset_index(drop=True); p=np.asarray(v2_meta_prob,float)
    pair_to_label,content_to_pos_titles,title_to_pos_contents=build_graph_reference(ref_df)
    rules=np.array(["semantic"]*len(q),dtype=object); forced=np.full(len(q),-1,dtype=np.int8)
    for pos,r in enumerate(q.itertuples()):
        if r.pair_key in pair_to_label:
            forced[pos]=int(pair_to_label[r.pair_key]); rules[pos]="exact_pair"; continue
        malformed=int(r.content_words)<MIN_GRAPH_CONTENT_WORDS
        pos_titles=content_to_pos_titles.get(r.content_key,set())
        if not malformed and len(pos_titles)==1 and r.title_key not in pos_titles:
            forced[pos]=0; rules[pos]="unique_content_different_title"; continue
        pos_contents=title_to_pos_contents.get(r.title_key,set())
        if not malformed and pos_contents and r.content_key not in pos_contents and p[pos]<V2_TITLE_GATE:
            forced[pos]=0; rules[pos]="gated_title_different_content"
    return rules,forced


def exact_macro_f1_threshold(y, probs):
    y=np.asarray(y,dtype=np.int8); p=np.asarray(probs,float)
    vals=np.unique(p); candidates=[np.nextafter(float(p.min()),-np.inf), *map(float,vals), np.nextafter(float(p.max()),np.inf)]
    best_t,best=-1.0,-1.0
    for t in candidates:
        s=f1_score(y,(p>=t).astype(np.int8),average="macro",zero_division=0)
        if s>best+1e-15 or (abs(s-best)<=1e-15 and abs(t-.5)<abs(best_t-.5)):
            best_t,best=float(t),float(s)
    return best_t,best

v2_oof_meta=v2_oof.meta_prob.to_numpy(float); v2_test_meta=v2_test.meta_prob.to_numpy(float)
v2_oof_global=v2_oof.global_e.to_numpy(float); v2_test_global=v2_test.global_e.to_numpy(float)
v3_oof_blend=v3_oof.blend_prob.to_numpy(float); v3_test_blend=v3_test.blend_prob.to_numpy(float)
rank_oof=RANK_V3_WEIGHT*v3_oof_blend+RANK_V2_GLOBAL_WEIGHT*v2_oof_global
rank_test=RANK_V3_WEIGHT*v3_test_blend+RANK_V2_GLOBAL_WEIGHT*v2_test_global
V2_META_THRESHOLD,_=exact_macro_f1_threshold(train.label.to_numpy(np.int8),v2_oof_meta)


def base_v2_for_query(query_df, ref_df, meta_prob):
    semantic=(np.asarray(meta_prob,float)>=V2_META_THRESHOLD).astype(np.int8)
    rules,forced=frozen_v2_graph_decisions(query_df,ref_df,meta_prob)
    labels=semantic.copy(); m=forced>=0; labels[m]=forced[m]
    return labels,rules,forced


def apply_transductive_v4(query_df, base_labels, base_forced, rank_scores, ref_df):
    q=query_df.reset_index(drop=True).copy(); labels=np.asarray(base_labels,np.int8).copy(); base_forced=np.asarray(base_forced,np.int8)
    rank_scores=np.asarray(rank_scores,float); n=len(q)
    _,content_to_pos_titles,title_to_pos_contents=build_graph_reference(ref_df)
    v4_forced=np.full(n,-1,dtype=np.int8); v4_rule=np.array(["v2_fallback"]*n,dtype=object)
    locked=base_forced>=0; v4_rule[locked]="v2_graph_locked"; rec=[]
    available=np.where(~locked)[0]
    if len(available):
        aq=q.iloc[available].copy(); aq["_pos"]=available
        for content_key,g in aq.groupby("content_key",sort=False):
            if int(g.content_words.iloc[0])<MIN_GRAPH_CONTENT_WORDS or g.title_key.nunique()<2: continue
            if len(content_to_pos_titles.get(content_key,set()))>0: continue
            cand=(g.assign(_score=rank_scores[g._pos.to_numpy(int)]).groupby("title_key",as_index=False)
                  .agg(score=("_score","mean"),rows=("_pos","size"))
                  .sort_values(["score","title_key"],ascending=[False,True],kind="mergesort").reset_index(drop=True))
            top=float(cand.score.iloc[0])
            if top<CONTENT_CANONICAL_MIN_SCORE:
                rec.append({"axis":"content","group_key":content_key,"top_score":top,"applied":False}); continue
            key=cand.title_key.iloc[0]; idx=g._pos.to_numpy(int); new=(g.title_key.to_numpy()==key).astype(np.int8)
            labels[idx]=new; v4_forced[idx]=new; v4_rule[idx]="test_content_canonical"; locked[idx]=True
            rec.append({"axis":"content","group_key":content_key,"top_score":top,"applied":True})
    available=np.where(~locked)[0]
    if len(available):
        aq=q.iloc[available].copy(); aq["_pos"]=available
        for title_key,g in aq.groupby("title_key",sort=False):
            if g.content_key.nunique()<2 or len(title_to_pos_contents.get(title_key,set()))>0: continue
            cand=(g.assign(_score=rank_scores[g._pos.to_numpy(int)]).groupby("content_key",as_index=False)
                  .agg(score=("_score","mean"),rows=("_pos","size"))
                  .sort_values(["score","content_key"],ascending=[False,True],kind="mergesort").reset_index(drop=True))
            top=float(cand.score.iloc[0]); second=float(cand.score.iloc[1]); margin=top-second
            if top<TITLE_CANONICAL_MIN_SCORE or margin<TITLE_CANONICAL_MIN_MARGIN:
                rec.append({"axis":"title","group_key":title_key,"top_score":top,"margin":margin,"applied":False}); continue
            key=cand.content_key.iloc[0]; idx=g._pos.to_numpy(int); new=(g.content_key.to_numpy()==key).astype(np.int8)
            labels[idx]=new; v4_forced[idx]=new; v4_rule[idx]="test_title_canonical"; locked[idx]=True
            rec.append({"axis":"title","group_key":title_key,"top_score":top,"margin":margin,"applied":True})
    immutable=base_forced>=0
    if not np.array_equal(labels[immutable],np.asarray(base_labels,np.int8)[immutable]): raise RuntimeError("V4 structural override violation")
    return labels,v4_rule,v4_forced,pd.DataFrame(rec)


def apply_v5_rescue(v4_labels,v4_rule,v3_blend,v2_global_e):
    labels=np.asarray(v4_labels,np.int8).copy(); rules=np.asarray(v4_rule,dtype=object)
    eligible=(labels==0)&(rules=="v2_fallback")&(np.asarray(v3_blend,float)>=V5_BLEND_MIN)&(np.asarray(v2_global_e,float)>=V5_GLOBAL_ENTAIL_MIN)
    labels[eligible]=1
    return labels,eligible

# Exact frozen V5 test parity: V6 will use this as the immutable champion base.
assert np.array_equal(v5_test.label.to_numpy(np.int8),v5_submission.label.to_numpy(np.int8))
print("Frozen V5 test distribution:", pd.Series(v5_test.label).value_counts().sort_index().to_dict())

IDF corpus:   0%|          | 0/14397 [00:00<?, ?it/s]

Frozen V5 test distribution: {0: 379, 1: 3224}


## Connected-component folds and strict frozen-V5 OOF reconstruction

All train-time label use remains inside the reference components of each outer fold. Exact normalized title/content components cannot cross a train/validation boundary.

In [5]:
class UnionFind:
    def __init__(self,n): self.p=list(range(n)); self.r=[0]*n
    def find(self,x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        ra,rb=self.find(a),self.find(b)
        if ra==rb:return
        if self.r[ra]<self.r[rb]:ra,rb=rb,ra
        self.p[rb]=ra
        if self.r[ra]==self.r[rb]:self.r[ra]+=1

def build_component_groups(df):
    uf=UnionFind(len(df))
    for col in ["title_key","content_key"]:
        first={}
        for i,key in enumerate(df[col].tolist()):
            if key in first: uf.union(i,first[key])
            else:first[key]=i
    roots=[uf.find(i) for i in range(len(df))]; mapping={r:j for j,r in enumerate(dict.fromkeys(roots))}
    return np.asarray([mapping[r] for r in roots],dtype=np.int32)

groups=build_component_groups(train)
cv=StratifiedGroupKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
fold_id=np.full(len(train),-1,dtype=np.int8)
oof_v5_label=np.full(len(train),-1,dtype=np.int8)
oof_v4_rule=np.array([""]*len(train),dtype=object)
oof_v5_rescue=np.zeros(len(train),dtype=bool)
fold_splits=[]
for fold,(tr_idx,va_idx) in enumerate(cv.split(train,train.label.values,groups=groups)):
    assert set(groups[tr_idx]).isdisjoint(set(groups[va_idx]))
    fold_id[va_idx]=fold; fold_splits.append((tr_idx,va_idx))
    ref=train.iloc[tr_idx].reset_index(drop=True); qry=train.iloc[va_idx].reset_index(drop=True)
    base,_,forced=base_v2_for_query(qry,ref,v2_oof_meta[va_idx])
    v4,v4_rules,_,_=apply_transductive_v4(qry,base,forced,rank_oof[va_idx],ref)
    v5,rescue=apply_v5_rescue(v4,v4_rules,v3_oof_blend[va_idx],v2_oof_global[va_idx])
    oof_v5_label[va_idx]=v5; oof_v4_rule[va_idx]=v4_rules; oof_v5_rescue[va_idx]=rescue
assert (fold_id>=0).all() and (oof_v5_label>=0).all()

# Core weak-supervision population: frozen V2 graph-unresolved rows only.
residual_mask=(v2_oof.graph_like_rule.astype(str).to_numpy()=="semantic")
y=train.label.to_numpy(np.int8)
hard_negative_mask=residual_mask & (y==0) & (v2_oof.meta_label.to_numpy(np.int8)==1)
print("Residual population:", int(residual_mask.sum()), "rows")
print("Residual negatives:", int((residual_mask&(y==0)).sum()))
print("V2 hard residual false-positive negatives:", int(hard_negative_mask.sum()))
print("OOF V5-positive pure-fallback candidate rows:", int(((oof_v5_label==1)&(oof_v4_rule=="v2_fallback")&(~oof_v5_rescue)).sum()))

Residual population: 9834 rows
Residual negatives: 490
V2 hard residual false-positive negatives: 138
OOF V5-positive pure-fallback candidate rows: 11687


## Real counterfactual slot supervision

Mine all high-similarity one-edit same-content canonical↔corrupted title pairs. Unlike V6B, V6C materializes **replace, delete, and insert** edits as an explicit `<FACT_SLOT>` with canonical and corrupted candidate states. `<NONE>` is a real candidate state for polarity insertion/deletion.

In [6]:
STOPWORDS=set("yang dan di ke dari untuk pada dengan adalah itu ini dalam akan telah sudah juga atau sebagai karena agar saat oleh atas terhadap antara tentang setelah sebelum lebih paling bisa dapat masih para sebuah suatu jadi yakni yaitu ada ia dia mereka kami kita anda kamu nya pun per serta hingga tanpa kalau jika namun tetapi sementara ketika dimana mana apa siapa bagaimana mengapa".split())
NEGATION_TOKENS={"tidak","tak","belum","bukan","jangan"}
NUMBER_UNITS=set("persen % ribu juta miliar triliun orang kasus pasien anak hari pekan minggu bulan tahun jam menit detik dosis unit kali km kg ton rp usd dolar".split())
NONE_TOKEN="<NONE>"
SLOT_TOKEN="<FACT_SLOT>"
HIGH_IDF_THRESHOLD=float(np.quantile(np.asarray(list(idf.values()),float),HIGH_IDF_QUANTILE))

def model_words(text): return TOKEN_RE.findall(clean_title_for_model(text))
def norm_token(w): return normalize_key(w)
def has_digit(text): return any(ch.isdigit() for ch in str(text))

def infer_slot_type(a,b):
    aa=normalize_key(a); bb=normalize_key(b)
    toks=set(aa.split())|set(bb.split())
    if has_digit(a) or has_digit(b): return "number"
    if a==NONE_TOKEN or b==NONE_TOKEN or bool(toks & NEGATION_TOKENS): return "polarity"
    return "lexical"

def align_counterfactual_titles(pos_title,neg_title):
    a=model_words(pos_title); b=model_words(neg_title)
    al=[x.lower() for x in a]; bl=[x.lower() for x in b]
    sm=difflib.SequenceMatcher(a=al,b=bl,autojunk=False)
    sim=float(sm.ratio()); edits=[op for op in sm.get_opcodes() if op[0]!="equal"]
    if sim<PAIR_SIMILARITY_MIN or len(edits)!=1: return None
    tag,i1,i2,j1,j2=edits[0]
    apos=a[i1:i2]; bneg=b[j1:j2]
    if max(len(apos),len(bneg))==0: return None
    if len(apos)>MAX_CHANGED_SPAN_TOKENS or len(bneg)>MAX_CHANGED_SPAN_TOKENS: return None
    canonical=" ".join(apos) if apos else NONE_TOKEN
    corrupted=" ".join(bneg) if bneg else NONE_TOKEN
    if tag=="replace": slot_words=a[:i1]+[SLOT_TOKEN]+a[i2:]
    elif tag=="delete": slot_words=a[:i1]+[SLOT_TOKEN]+a[i2:]
    elif tag=="insert": slot_words=a[:i1]+[SLOT_TOKEN]+a[i1:]
    else: return None
    slot_context=" ".join(slot_words)
    # Corrupted-side position is required to simulate inference on the wrong headline.
    corrupted_start=j1; corrupted_end=j2
    return {
        "similarity":sim,"edit":tag,"pos_i1":i1,"pos_i2":i2,"neg_j1":j1,"neg_j2":j2,
        "pos_span":" ".join(apos),"neg_span":" ".join(bneg),"canonical":canonical,"corrupted":corrupted,
        "slot_context":slot_context,"slot_type":infer_slot_type(canonical,corrupted),
        "corrupted_start":corrupted_start,"corrupted_end":corrupted_end,
    }

def mine_clean_counterfactual_pairs(df):
    rows=[]
    for ckey,g in df.groupby("content_key",sort=False):
        pos=g[g.label==1]; neg=g[g.label==0]
        pos_keys=pos.title_key.unique().tolist()
        if len(pos_keys)!=1 or len(neg)==0: continue
        pg=pos[pos.title_key==pos_keys[0]]; pidx=int(pg.index[0]); ptitle=str(pg.iloc[0].title_model)
        for nkey,ng in neg.groupby("title_key",sort=False):
            nidx=int(ng.index[0]); ntitle=str(ng.iloc[0].title_model)
            info=align_counterfactual_titles(ptitle,ntitle)
            if info is None: continue
            rows.append({"pair_id":len(rows),"content_key":ckey,"pos_idx":pidx,"neg_idx":nidx,"pos_title":ptitle,"neg_title":ntitle,**info})
    return pd.DataFrame(rows)

clean_pairs=mine_clean_counterfactual_pairs(train)
if len(clean_pairs)==0: raise RuntimeError("No clean counterfactual pairs mined; alignment drifted.")
clean_pairs["fold"]=fold_id[clean_pairs.pos_idx.to_numpy(int)]
assert np.array_equal(clean_pairs.fold.to_numpy(),fold_id[clean_pairs.neg_idx.to_numpy(int)])
print("Counterfactual pairs:",len(clean_pairs))
print(clean_pairs.groupby(["fold","edit"]).size().unstack(fill_value=0))
print(clean_pairs.groupby("slot_type").size())

Counterfactual pairs: 601
edit  delete  insert  replace
fold                         
0         36       6      152
1         41       3      162
2         34       9      158
slot_type
lexical     431
number       41
polarity    129
dtype: int64


## Shared slot evidence and article-grounded challenger generation

All candidates for one slot use exactly the same masked-context evidence. Lexical/number challengers come from those article windows; polarity insertion candidates are learned only from reference-fold counterfactual patterns. Generator recall is measured independently before any final veto is accepted.

In [7]:
def make_slot_context(title,start,end):
    ws=model_words(title); start=int(start); end=int(end)
    start=max(0,min(start,len(ws))); end=max(start,min(end,len(ws)))
    return " ".join(ws[:start]+[SLOT_TOKEN]+ws[end:])

def slot_query_tokens(slot_context):
    return [t for t in tokens(slot_context.replace(SLOT_TOKEN," ")) if t]

def retrieve_shared_evidence(slot_context,content):
    qt=slot_query_tokens(slot_context)
    windows=make_windows(content,SLOT_WINDOW_WORDS,SLOT_WINDOW_STRIDE)
    scored=[]
    for wi,(_,txt) in enumerate(windows): scored.append((title_window_score(qt,txt),-wi,wi))
    scored.sort(reverse=True)
    chosen=[x[2] for x in scored[:SLOT_TOP_WINDOWS]] or [0]
    evid=[windows[i][1] for i in chosen]
    while len(evid)<SLOT_TOP_WINDOWS: evid.append(evid[-1])
    return evid

def candidate_claim(slot_context,candidate):
    c="<NONE> (tanpa token tambahan)" if candidate==NONE_TOKEN else candidate
    return f"Konteks judul: {slot_context}. Kandidat untuk <FACT_SLOT>: {c}."

def normalize_candidate(x):
    return NONE_TOKEN if x==NONE_TOKEN else normalize_key(x)

def exact_candidate_in_evidence(candidate,evidence):
    if candidate==NONE_TOKEN: return True
    ck=normalize_key(candidate)
    return any(ck and ck in normalize_key(ev) for ev in evidence)

def local_context_score(candidate_tokens, words, start, context_tokens, radius=8):
    lo=max(0,start-radius); hi=min(len(words),start+len(candidate_tokens)+radius)
    nbh=set(normalize_key(x) for x in words[lo:hi] if normalize_key(x))
    cset=set(context_tokens)
    return sum(tok_idf(t) for t in cset if t in nbh)

def extract_numeric_candidates(evidence,slot_context,original):
    context_toks=slot_query_tokens(slot_context); best={}
    for wrank,ev in enumerate(evidence):
        ws=model_words(ev); lows=[normalize_key(x) for x in ws]
        for i,w in enumerate(lows):
            if not has_digit(w): continue
            j=i+1
            if j<len(lows) and lows[j] in NUMBER_UNITS: j+=1
            text=" ".join(ws[i:j]); key=normalize_candidate(text)
            if not key or key==normalize_candidate(original): continue
            score=20.0-2.0*wrank+local_context_score(lows[i:j],ws,i,context_toks)+0.05*sum(tok_idf(t) for t in lows[i:j])
            if key not in best or score>best[key][0]: best[key]=(score,text)
    vals=sorted(best.values(),key=lambda z:(-z[0],normalize_key(z[1])))
    return [v[1] for v in vals[:MAX_CHALLENGERS]]

def extract_lexical_candidates(evidence,slot_context,original):
    context_toks=slot_query_tokens(slot_context); olen=max(1,len(tokens(original))) if original!=NONE_TOKEN else 1; best={}
    for wrank,ev in enumerate(evidence):
        ws=model_words(ev); lows=[normalize_key(x) for x in ws]
        for n in (1,2,3):
            if abs(n-olen)>1: continue
            for i in range(0,len(ws)-n+1):
                seg=lows[i:i+n]
                if not any(seg) or all(x in STOPWORDS for x in seg): continue
                if any(has_digit(x) for x in seg): continue
                if max((tok_idf(x) for x in seg if x),default=0)<HIGH_IDF_THRESHOLD: continue
                text=" ".join(ws[i:i+n]); key=normalize_candidate(text)
                if not key or key==normalize_candidate(original): continue
                overlap=local_context_score(seg,ws,i,context_toks)
                score=20.0-2.0*wrank+overlap+0.05*sum(tok_idf(t) for t in seg)-0.30*abs(n-olen)
                if key not in best or score>best[key][0]: best[key]=(score,text)
    vals=sorted(best.values(),key=lambda z:(-z[0],normalize_key(z[1])))
    return [v[1] for v in vals[:MAX_CHALLENGERS]]

def polarity_anchor(slot_context):
    ws=model_words(slot_context.replace(SLOT_TOKEN," FACTSLOT "))
    try: p=[normalize_key(x) for x in ws].index("factslot")
    except ValueError: return ("","")
    left=normalize_key(ws[p-1]) if p>0 else "<BOS>"
    right=normalize_key(ws[p+1]) if p+1<len(ws) else "<EOS>"
    return left,right

def learn_polarity_patterns(pair_records):
    exact=defaultdict(Counter); right=defaultdict(Counter)
    for p in pair_records:
        if p["slot_type"]!="polarity": continue
        left,rgt=polarity_anchor(p["slot_context"])
        vals=[p["canonical"],p["corrupted"]]
        for v in vals:
            if v==NONE_TOKEN: continue
            vv=normalize_key(v)
            if vv in NEGATION_TOKENS:
                exact[(left,rgt)][vv]+=1; right[rgt][vv]+=1
    return {"exact":exact,"right":right}

def polarity_insert_slots(title,patterns,max_slots=MAX_POLARITY_SLOTS):
    ws=model_words(title); lows=[normalize_key(x) for x in ws]; rec=[]
    for pos in range(len(ws)+1):
        left=lows[pos-1] if pos>0 else "<BOS>"; right=lows[pos] if pos<len(ws) else "<EOS>"
        cnt=patterns["exact"].get((left,right),Counter())
        if not cnt: cnt=patterns["right"].get(right,Counter())
        if not cnt: continue
        challengers=[x for x,_ in cnt.most_common(MAX_CHALLENGERS)]
        support=sum(cnt.values())
        rec.append((support,pos,challengers))
    rec.sort(key=lambda z:(-z[0],z[1]))
    return rec[:max_slots]

def extract_title_slots(title):
    ws=model_words(title); lows=[normalize_key(x) for x in ws]
    occupied=set(); nums=[]; pol=[]; lex=[]
    for i,w in enumerate(lows):
        if has_digit(w):
            j=i+1
            if j<len(lows) and lows[j] in NUMBER_UNITS: j+=1
            nums.append({"start":i,"end":j,"original":" ".join(ws[i:j]),"type":"number"}); occupied.update(range(i,j))
        if w in NEGATION_TOKENS:
            pol.append({"start":i,"end":i+1,"original":ws[i],"type":"polarity"}); occupied.add(i)
    # High-information lexical slots, deliberately few to control test inference cost.
    cand=[]
    for n in (1,2,3):
        for i in range(0,len(ws)-n+1):
            if any(j in occupied for j in range(i,i+n)): continue
            seg=lows[i:i+n]
            if not any(seg) or all(x in STOPWORDS for x in seg) or any(has_digit(x) for x in seg): continue
            vals=[tok_idf(x) for x in seg if x]
            if not vals or max(vals)<HIGH_IDF_THRESHOLD: continue
            cand.append((sum(vals)+0.05*n,i,n," ".join(ws[i:i+n])))
    cand.sort(key=lambda z:(-z[0],z[1],z[2]))
    used=[]
    for score,i,n,text in cand:
        if any(set(range(i,i+n)) & set(range(u["start"],u["end"])) for u in used): continue
        used.append({"start":i,"end":i+n,"original":text,"type":"lexical","priority":score})
        if len(used)>=MAX_LEXICAL_SLOTS: break
    return nums[:MAX_NUMERIC_SLOTS]+pol[:MAX_POLARITY_SLOTS]+used

def build_slot_record(title,content,start,end,original,slot_type,patterns=None,forced_challengers=None):
    context=make_slot_context(title,start,end); evidence=retrieve_shared_evidence(context,content)
    if forced_challengers is not None:
        challengers=list(forced_challengers)
    elif slot_type=="number": challengers=extract_numeric_candidates(evidence,context,original)
    elif slot_type=="lexical": challengers=extract_lexical_candidates(evidence,context,original)
    elif slot_type=="polarity": challengers=[NONE_TOKEN] if original!=NONE_TOKEN else []
    else: challengers=[]
    seen={normalize_candidate(original)}; ch=[]
    for c in challengers:
        k=normalize_candidate(c)
        if not k or k in seen: continue
        seen.add(k); ch.append(c)
        if len(ch)>=MAX_CHALLENGERS: break
    if not ch: return None
    return {"slot_id":f"{slot_type}:{start}:{end}:{normalize_key(context)}","type":slot_type,"start":int(start),"end":int(end),"context":context,"evidence":evidence,"original":original,"challengers":ch}

def build_row_candidate_slots(row,patterns):
    slots=[]; existing=set()
    for s in extract_title_slots(row.title_model):
        rec=build_slot_record(row.title_model,row.content,s["start"],s["end"],s["original"],s["type"],patterns)
        if rec is not None and rec["slot_id"] not in existing: slots.append(rec); existing.add(rec["slot_id"])
    # Missing-negation candidates learned strictly from the supplied reference patterns.
    explicit_positions={s["start"] for s in extract_title_slots(row.title_model) if s["type"]=="polarity"}
    for _,pos,ch in polarity_insert_slots(row.title_model,patterns):
        if pos in explicit_positions: continue
        rec=build_slot_record(row.title_model,row.content,pos,pos,NONE_TOKEN,"polarity",patterns,forced_challengers=ch)
        if rec is not None and rec["slot_id"] not in existing: slots.append(rec); existing.add(rec["slot_id"])
    return slots

def pair_dicts(df_pairs): return [r._asdict() for r in df_pairs.itertuples(index=False)]
ALL_PAIR_RECORDS=pair_dicts(clean_pairs)

## Candidate-ranker model and AMP-safe training primitives

AMP is restricted to transformer forward passes. Softmax/BCE/margin arithmetic is explicitly FP32, preventing the autocast/BCE failure encountered in the first V6B draft.

In [8]:
def resolve_model_source():
    if LOCAL_MODEL_PATH: return LOCAL_MODEL_PATH
    root=Path("/kaggle/input")
    if root.exists():
        for cfg in root.rglob("config.json"):
            p=cfg.parent; name=str(p).lower()
            if "mdeberta" in name and (p/"tokenizer_config.json").exists():
                print("Using local Kaggle mDeBERTa model:",p); return str(p)
    return MODEL_NAME

def ensure_fp32_master_weights(model,verbose=True):
    bad=[p for p in model.parameters() if p.is_floating_point() and p.dtype!=torch.float32]
    if bad: model.float()
    rem={str(p.dtype) for p in model.parameters() if p.is_floating_point() and p.dtype!=torch.float32}
    if rem: raise RuntimeError(f"FP32 master-weight invariant failed: {sorted(rem)}")
    if verbose: print("Master parameter dtypes:",sorted({str(p.dtype) for p in model.parameters() if p.is_floating_point()}))
    return model

def model_kwargs(source): return {} if LOCAL_MODEL_PATH or source!=MODEL_NAME else {"revision":MODEL_REVISION}

def load_candidate_model():
    source=resolve_model_source(); kw=model_kwargs(source)
    tok=AutoTokenizer.from_pretrained(source,use_fast=True,**kw)
    try:
        model=AutoModelForSequenceClassification.from_pretrained(source,num_labels=2,ignore_mismatched_sizes=True,dtype=torch.float32,**kw)
    except TypeError:
        model=AutoModelForSequenceClassification.from_pretrained(source,num_labels=2,ignore_mismatched_sizes=True,torch_dtype=torch.float32,**kw)
    model=ensure_fp32_master_weights(model)
    model.config.id2label={0:"UNSUPPORTED",1:"SUPPORTED"}; model.config.label2id={"UNSUPPORTED":0,"SUPPORTED":1}; model.config.use_cache=False
    return tok,model,source

def unwrap_model(model): return model.module if isinstance(model,nn.DataParallel) else model
def amp_context(): return torch.autocast("cuda",dtype=torch.float16) if AMP_ENABLED else nullcontext()
def make_grad_scaler():
    try: return torch.amp.GradScaler("cuda",enabled=AMP_ENABLED)
    except (AttributeError,TypeError): return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
def loader_kwargs(): return {"num_workers":NUM_WORKERS,"pin_memory":DEVICE.type=="cuda","persistent_workers":NUM_WORKERS>0}
def to_device(enc): return {k:v.to(DEVICE,non_blocking=True) for k,v in enc.items()}
def support_probs(logits): return torch.softmax(logits.float(),dim=-1)[:,1]

def encode_sequences(tok,evidences,claims):
    return tok(evidences,claims,padding=True,truncation="only_first",max_length=MAX_LENGTH,return_tensors="pt")

class IndexDataset(Dataset):
    def __init__(self,idx): self.idx=list(idx)
    def __len__(self): return len(self.idx)
    def __getitem__(self,i): return self.idx[i]

def collate_list(xs): return xs

def flatten_candidate(candidate,context,evidence,evid_out,claim_out):
    positions=[]; claim=candidate_claim(context,candidate)
    for ev in evidence:
        positions.append(len(evid_out)); evid_out.append(ev); claim_out.append(claim)
    return positions

def aggregate_max(seq_support,groups):
    vals=[]
    for g in groups:
        ix=torch.as_tensor(g,device=seq_support.device,dtype=torch.long); vals.append(seq_support[ix].max())
    return torch.stack(vals) if vals else torch.empty(0,device=seq_support.device,dtype=torch.float32)

## Fold-safe positive controls and training schedule

Every graph-unresolved positive reference row is considered for a deterministic article-verifiable positive control. No negative row-level MIL is used. Each optimization step combines a batch of positive controls with one real counterfactual pair.

In [9]:
def build_positive_control_for_row(row):
    # One deterministic high-quality positive candidate per row; no label is used to construct the slot itself.
    for s in extract_title_slots(row.title_model):
        context=make_slot_context(row.title_model,s["start"],s["end"]); evidence=retrieve_shared_evidence(context,row.content)
        if exact_candidate_in_evidence(s["original"],evidence):
            return {"context":context,"evidence":evidence,"candidate":s["original"],"type":s["type"],"slot_id":f"{s['type']}:{s['start']}:{s['end']}:{normalize_key(context)}"}
    return None

print("Precomputing one label-free positive-control candidate per train row...")
positive_control_bank=[build_positive_control_for_row(r) for r in tqdm(train.itertuples(),total=len(train),desc="positive controls")]
print("Rows with a verifiable candidate:",sum(x is not None for x in positive_control_bank))

def materialize_training_pair(p):
    content=str(train.loc[int(p["pos_idx"]),"content"])
    context=p["slot_context"]; evidence=retrieve_shared_evidence(context,content)
    return {**p,"evidence":evidence}

STRONG_PAIRS=[materialize_training_pair(p) for p in ALL_PAIR_RECORDS]
MODEL_SOURCE=None
fold_history=[]; fold_diag_rows=[]; generator_rows=[]

def score_candidate_set(model,tok,slot):
    candidates=[slot["original"]]+list(slot["challengers"]); evid=[]; claims=[]; groups=[]
    for c in candidates: groups.append(flatten_candidate(c,slot["context"],slot["evidence"],evid,claims))
    vals=[]
    for st in range(0,len(evid),EVAL_SEQUENCE_BATCH):
        enc=to_device(encode_sequences(tok,evid[st:st+EVAL_SEQUENCE_BATCH],claims[st:st+EVAL_SEQUENCE_BATCH]))
        with torch.no_grad(), amp_context(): logits=model(**enc,return_dict=False)[0]
        vals.extend(support_probs(logits).detach().cpu().numpy().tolist()); del enc,logits
    seq=torch.tensor(vals,dtype=torch.float32)
    cand=aggregate_max(seq,groups).numpy()
    return candidates,cand

def strong_pair_validation_slot(p,patterns):
    # Simulate deployment on the corrupted headline, not an oracle two-candidate input.
    neg_words=model_words(p["neg_title"]); start=int(p["neg_j1"]); end=int(p["neg_j2"])
    original=p["corrupted"]
    if p["edit"]=="delete": start=end=int(p["neg_j1"]); original=NONE_TOKEN
    slot_type=p["slot_type"]
    context=make_slot_context(p["neg_title"],start,end)
    evidence=retrieve_shared_evidence(context,str(train.loc[int(p["neg_idx"]),"content"]))
    if slot_type=="polarity":
        if original!=NONE_TOKEN: challengers=[NONE_TOKEN]
        else:
            left,right=polarity_anchor(context); cnt=patterns["exact"].get((left,right),Counter()) or patterns["right"].get(right,Counter())
            challengers=[x for x,_ in cnt.most_common(MAX_CHALLENGERS)]
        rec=build_slot_record(p["neg_title"],str(train.loc[int(p["neg_idx"]),"content"]),start,end,original,slot_type,patterns,forced_challengers=challengers)
    else:
        rec=build_slot_record(p["neg_title"],str(train.loc[int(p["neg_idx"]),"content"]),start,end,original,slot_type,patterns)
    return rec

def train_fold(fold,tr_idx,va_idx):
    global MODEL_SOURCE
    ref_pairs=[p for p in STRONG_PAIRS if int(p["fold"])!=fold]
    val_pairs=[p for p in STRONG_PAIRS if int(p["fold"])==fold]
    ref_pair_raw=[p for p in ALL_PAIR_RECORDS if int(p["fold"])!=fold]
    patterns=learn_polarity_patterns(ref_pair_raw)
    controls=[i for i in tr_idx if residual_mask[i] and y[i]==1 and positive_control_bank[i] is not None]
    if not ref_pairs or not controls: raise RuntimeError(f"Fold {fold}: empty strong-pair/control training source")

    tok,base_model,source=load_candidate_model(); MODEL_SOURCE=source
    base_model.to(DEVICE); ensure_fp32_master_weights(base_model)
    model=nn.DataParallel(base_model) if GPU_COUNT>=2 else base_model
    if sorted({str(p.dtype) for p in unwrap_model(model).parameters() if p.is_floating_point()}) != ["torch.float32"]:
        raise RuntimeError("Optimizer master parameters are not FP32")

    gen=torch.Generator(); gen.manual_seed(SEED+fold)
    loader=DataLoader(IndexDataset(controls),batch_size=CONTROL_BATCH_SIZE,shuffle=True,generator=gen,collate_fn=collate_list,**loader_kwargs())
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scaler=make_grad_scaler(); updates=math.ceil(len(loader)/GRAD_ACCUM)*EPOCHS
    sched=get_linear_schedule_with_warmup(opt,int(updates*WARMUP_RATIO),updates); opt.zero_grad(set_to_none=True)

    for epoch in range(EPOCHS):
        model.train(); rng=np.random.default_rng(SEED+1000*fold+epoch); order=rng.permutation(len(ref_pairs)); losses=[]; lbce=[]; lrank=[]; lctrl=[]; t0=time.perf_counter()
        pbar=tqdm(loader,desc=f"V6C fold {fold} epoch {epoch+1}/{EPOCHS}")
        for step,control_idx in enumerate(pbar):
            pair=ref_pairs[int(order[step%len(order)])]
            evid=[]; claims=[]
            pair_groups=[]
            for cand in [pair["canonical"],pair["corrupted"]]: pair_groups.append(flatten_candidate(cand,pair["slot_context"],pair["evidence"],evid,claims))
            control_groups=[]
            for idx in control_idx:
                c=positive_control_bank[int(idx)]; control_groups.append(flatten_candidate(c["candidate"],c["context"],c["evidence"],evid,claims))
            enc=to_device(encode_sequences(tok,evid,claims))
            # AMP ONLY for transformer forward. All probability-domain losses remain FP32.
            with amp_context(): logits=model(**enc,return_dict=False)[0]
            seq_support=support_probs(logits)
            pair_support=aggregate_max(seq_support,pair_groups).clamp(1e-6,1-1e-6)
            pair_labels=torch.tensor([1.0,0.0],device=DEVICE,dtype=torch.float32)
            candidate_bce=F.binary_cross_entropy(pair_support,pair_labels)
            pair_loss=F.relu(PAIR_MARGIN-(pair_support[0]-pair_support[1]))
            if control_groups:
                ctrl_support=aggregate_max(seq_support,control_groups).clamp(1e-6,1-1e-6)
                ctrl_loss=F.binary_cross_entropy(ctrl_support,torch.ones_like(ctrl_support))
            else: ctrl_loss=torch.zeros((),device=DEVICE,dtype=torch.float32)
            loss=CANDIDATE_BCE_WEIGHT*candidate_bce+PAIR_LOSS_WEIGHT*pair_loss+POSITIVE_CONTROL_WEIGHT*ctrl_loss
            scaler.scale(loss/GRAD_ACCUM).backward()
            losses.append(float(loss.detach())); lbce.append(float(candidate_bce.detach())); lrank.append(float(pair_loss.detach())); lctrl.append(float(ctrl_loss.detach()))
            if ((step+1)%GRAD_ACCUM==0) or (step+1==len(loader)):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),MAX_GRAD_NORM)
                bad=[p.grad.dtype for p in unwrap_model(model).parameters() if p.grad is not None and p.grad.dtype!=torch.float32]
                if bad: raise RuntimeError(f"Non-FP32 master gradients detected: {set(map(str,bad))}")
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            if step%50==0: pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.3f}",pair=f"{np.mean(lbce[-50:]):.3f}",ctrl=f"{np.mean(lctrl[-50:]):.3f}")
            del enc,logits,seq_support,loss
        fold_history.append({"fold":fold,"epoch":epoch+1,"loss":float(np.mean(losses)),"candidate_bce":float(np.mean(lbce)),"pair_margin_loss":float(np.mean(lrank)),"positive_control_loss":float(np.mean(lctrl)),"seconds":time.perf_counter()-t0,"controls":len(controls),"strong_pairs":len(ref_pairs)})

    # Held-out generator + full candidate-pool ranking diagnostic.
    model.eval(); type_stats=defaultdict(lambda:{"n":0,"generated":0,"top1":0})
    for p in tqdm(val_pairs,desc=f"fold {fold} held-out generator/ranking"):
        rec=strong_pair_validation_slot(p,patterns); typ=p["slot_type"]; type_stats[typ]["n"]+=1
        canonical_key=normalize_candidate(p["canonical"])
        generated=False; top1=False; original_score=np.nan; canonical_score=np.nan; best_candidate=""; best_score=np.nan
        if rec is not None:
            cands,scores=score_candidate_set(model,tok,rec); keys=[normalize_candidate(x) for x in cands]
            generated=canonical_key in keys
            oi=0; original_score=float(scores[0])
            bi=int(np.argmax(scores)); best_candidate=cands[bi]; best_score=float(scores[bi])
            if generated:
                ci=keys.index(canonical_key); canonical_score=float(scores[ci]); top1=(bi==ci)
        type_stats[typ]["generated"]+=int(generated); type_stats[typ]["top1"]+=int(top1)
        generator_rows.append({"pair_id":p["pair_id"],"fold":fold,"edit":p["edit"],"slot_type":typ,"canonical":p["canonical"],"corrupted":p["corrupted"],"generated":int(generated),"top1_correct":int(top1),"original_score":original_score,"canonical_score":canonical_score,"best_candidate":best_candidate,"best_score":best_score})

    fold_diag={"fold":fold,"strong_train_pairs":len(ref_pairs),"positive_controls":len(controls),"strong_val_pairs":len(val_pairs)}
    for typ in ["lexical","number","polarity"]:
        st=type_stats[typ]; fold_diag[f"{typ}_n"]=st["n"]; fold_diag[f"{typ}_generator_recall"]=st["generated"]/st["n"] if st["n"] else np.nan; fold_diag[f"{typ}_top1_all"]=st["top1"]/st["n"] if st["n"] else np.nan
    fold_diag["generator_recall"]=sum(v["generated"] for v in type_stats.values())/max(1,sum(v["n"] for v in type_stats.values()))
    fold_diag["coverage_adjusted_top1"]=sum(v["top1"] for v in type_stats.values())/max(1,sum(v["n"] for v in type_stats.values()))
    fold_diag["top1_given_generated"]=sum(v["top1"] for v in type_stats.values())/max(1,sum(v["generated"] for v in type_stats.values()))

    return model,tok,patterns,fold_diag

Precomputing one label-free positive-control candidate per train row...


positive controls:   0%|          | 0/14397 [00:00<?, ?it/s]

Rows with a verifiable candidate: 7369


## Candidate-bank scoring and strict OOF residual margins

The V6C row signal is **challenger advantage** (`best challenger support − original support`), stored separately for lexical/entity, numeric, and polarity slots. There is no absolute minimum-support veto.

In [10]:
TYPES=["lexical","number","polarity"]
oof_candidate=(oof_v5_label==1)&(oof_v4_rule=="v2_fallback")&(~oof_v5_rescue)
oof_margin={t:np.full(len(train),np.nan,dtype=float) for t in TYPES}
oof_best_slot={t:np.array([""]*len(train),dtype=object) for t in TYPES}
oof_original={t:np.array([""]*len(train),dtype=object) for t in TYPES}
oof_challenger={t:np.array([""]*len(train),dtype=object) for t in TYPES}

# Frozen V5 test candidate scope.
final_v5=v5_test.label.to_numpy(np.int8)
test_candidate=(final_v5==1)&(v5_test.v4_rule.astype(str).to_numpy()=="v2_fallback")&(~v5_test.v5_rescue.to_numpy(np.int8).astype(bool))

# Test banks must be identical across fold models; full-train polarity patterns are legal at test time.
FULL_PATTERNS=learn_polarity_patterns(ALL_PAIR_RECORDS)
print("Building common V6C test challenger bank for",int(test_candidate.sum()),"eligible rows...")
test_banks={}
for idx in tqdm(np.flatnonzero(test_candidate),desc="test challenger banks"):
    slots=build_row_candidate_slots(test.iloc[int(idx)],FULL_PATTERNS)
    if slots: test_banks[int(idx)]=slots

# fold -> row -> slot_id -> compact score record
TEST_MODEL_SLOT_RESULTS=[{} for _ in range(N_FOLDS)]
trained_models=[]

def score_bank_bulk(model,tok,bank_dict,desc,row_chunk=24):
    # Tokenizer/model calls are batched across many rows to keep T4x2 inference efficient.
    out={}; ids=list(bank_dict.keys())
    for st in tqdm(range(0,len(ids),row_chunk),desc=desc):
        chunk=ids[st:st+row_chunk]; evid=[]; claims=[]; metas=[]
        for idx in chunk:
            for slot in bank_dict[idx]:
                cands=[slot["original"]]+list(slot["challengers"]); cand_groups=[]
                for cand in cands:
                    g=flatten_candidate(cand,slot["context"],slot["evidence"],evid,claims); cand_groups.append(g)
                metas.append((idx,slot,cands,cand_groups))
        seq_scores=[]
        for bs in range(0,len(evid),EVAL_SEQUENCE_BATCH):
            enc=to_device(encode_sequences(tok,evid[bs:bs+EVAL_SEQUENCE_BATCH],claims[bs:bs+EVAL_SEQUENCE_BATCH]))
            with torch.no_grad(), amp_context(): logits=model(**enc,return_dict=False)[0]
            seq_scores.extend(support_probs(logits).detach().cpu().numpy().tolist()); del enc,logits
        seq=np.asarray(seq_scores,dtype=np.float32)
        for idx,slot,cands,groups in metas:
            scores=np.asarray([float(seq[np.asarray(g,dtype=int)].max()) for g in groups],dtype=float)
            if len(scores)<2: continue
            orig=float(scores[0]); j=1+int(np.argmax(scores[1:])); chall=float(scores[j]); margin=chall-orig
            out.setdefault(int(idx),[]).append({"slot_id":slot["slot_id"],"type":slot["type"],"original":slot["original"],"challenger":cands[j],"original_score":orig,"challenger_score":chall,"margin":margin})
    return out

fold_summaries=[]
for fold,(tr_idx,va_idx) in enumerate(fold_splits):
    seed_everything(SEED+fold)
    model,tok,patterns,diag=train_fold(fold,tr_idx,va_idx); fold_summaries.append(diag)
    model.eval()
    # Strict held-out residual scoring uses only fold-reference polarity patterns.
    va_eligible=np.asarray([i for i in va_idx if oof_candidate[i]],dtype=int)
    va_banks={}
    for idx in tqdm(va_eligible,desc=f"fold {fold} residual challenger banks"):
        slots=build_row_candidate_slots(train.iloc[int(idx)],patterns)
        if slots: va_banks[int(idx)]=slots
    va_results=score_bank_bulk(model,tok,va_banks,f"fold {fold} residual OOF scoring")
    for idx,res in va_results.items():
        for typ in TYPES:
            rr=[x for x in res if x["type"]==typ]
            if rr:
                best=max(rr,key=lambda x:(x["margin"],x["slot_id"],normalize_candidate(x["challenger"])))
                oof_margin[typ][idx]=best["margin"]; oof_best_slot[typ][idx]=best["slot_id"]; oof_original[typ][idx]=best["original"]; oof_challenger[typ][idx]=best["challenger"]
    # Score the exact same common test candidate bank with this fold model.
    raw_test=score_bank_bulk(model,tok,test_banks,f"fold {fold} test challenger scoring")
    TEST_MODEL_SLOT_RESULTS[fold]={idx:{x["slot_id"]:x for x in res} for idx,res in raw_test.items()}
    del model,tok; gc.collect(); torch.cuda.empty_cache()

fold_specialist_diag=pd.DataFrame(fold_summaries)
training_history=pd.DataFrame(fold_history)
generator_validation=pd.DataFrame(generator_rows)
print("Fold V6C diagnostics:"); display(fold_specialist_diag)

Building common V6C test challenger bank for 2255 eligible rows...


test challenger banks:   0%|          | 0/2255 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

...

fold 2 residual challenger banks:   0%|          | 0/3854 [00:00<?, ?it/s]

fold 2 residual OOF scoring:   0%|          | 0/129 [00:00<?, ?it/s]

fold 2 test challenger scoring:   0%|          | 0/78 [00:00<?, ?it/s]

Fold V6C diagnostics:


,fold,strong_train_pairs,positive_controls,strong_val_pairs,lexical_n,lexical_generator_recall,lexical_top1_all,number_n,number_generator_recall,number_top1_all,polarity_n,polarity_generator_recall,polarity_top1_all,generator_recall,coverage_adjusted_top1,top1_given_generated
0,0,407,3184,194,135,0.0,0.0,17,0.235294,0.0,42,0.309524,0.142857,0.087629,0.030928,0.352941
1,1,395,3198,206,151,0.0,0.0,11,0.272727,0.0,44,0.363636,0.272727,0.092233,0.058252,0.631579
2,2,400,3260,201,145,0.0,0.0,13,0.076923,0.0,43,0.255814,0.046512,0.059701,0.009950,0.166667


## Generator/ranker diagnostics and cross-fitted type-specific veto thresholds

A held-out row may be vetoed only when its challenger advantage exceeds a threshold learned from the other outer folds for that factual type. The primary evaluation is the union of those untouched held-out vetoes.

In [11]:
def choose_margin_threshold(scores,y_neg,min_precision=MIN_VETO_PRECISION,min_selected=MIN_THRESHOLD_SELECTED):
    s=np.asarray(scores,float); yv=np.asarray(y_neg,np.int8); m=np.isfinite(s); s=s[m]; yv=yv[m]
    if len(s)==0: return np.inf,{"selected":0,"precision":np.nan,"recall":0.0}
    order=np.argsort(-s,kind="mergesort"); ss=s[order]; yy=yv[order]
    tp=np.cumsum(yy); n=np.arange(1,len(yy)+1); prec=tp/n; total=max(1,int(yy.sum())); rec=tp/total
    valid=np.flatnonzero((n>=min_selected)&(prec>=min_precision))
    if len(valid)==0: return np.inf,{"selected":0,"precision":np.nan,"recall":0.0}
    # highest recall; tie -> fewer interventions / higher threshold
    best=sorted(valid,key=lambda i:(-rec[i],n[i],-ss[i]))[0]
    return float(ss[best]),{"selected":int(n[best]),"precision":float(prec[best]),"recall":float(rec[best])}

y_neg=(y==0).astype(np.int8)
fold_type_threshold={fold:{t:np.inf for t in TYPES} for fold in range(N_FOLDS)}
crossfit_type_veto={t:np.zeros(len(train),dtype=bool) for t in TYPES}; threshold_rows=[]
for fold in range(N_FOLDS):
    for typ in TYPES:
        fit=oof_candidate&(fold_id!=fold)&np.isfinite(oof_margin[typ]); hold=oof_candidate&(fold_id==fold)&np.isfinite(oof_margin[typ])
        thr,info=choose_margin_threshold(oof_margin[typ][fit],y_neg[fit]); fold_type_threshold[fold][typ]=thr
        sel=hold&(oof_margin[typ]>=thr) if np.isfinite(thr) else np.zeros(len(train),dtype=bool)
        crossfit_type_veto[typ]|=sel
        n=int(sel.sum()); tp=int(y_neg[sel].sum())
        threshold_rows.append({"fold":fold,"type":typ,"threshold":thr,"heldout_selected":n,"heldout_precision":tp/n if n else np.nan,"heldout_corrected_negatives":tp,"heldout_false_vetoes":n-tp,"fit_selected":info["selected"],"fit_precision":info["precision"],"fit_recall":info["recall"]})
threshold_metrics=pd.DataFrame(threshold_rows)
crossfit_veto=np.zeros(len(train),dtype=bool)
for t in TYPES: crossfit_veto|=crossfit_type_veto[t]

v6c_oof=oof_v5_label.copy(); v6c_oof[crossfit_veto]=0
component_rows=[]
for fold in range(N_FOLDS):
    m=fold_id==fold; f5=f1_score(y[m],oof_v5_label[m],average="macro",zero_division=0); f6=f1_score(y[m],v6c_oof[m],average="macro",zero_division=0)
    sel=crossfit_veto&m; n=int(sel.sum()); tp=int(y_neg[sel].sum())
    component_rows.append({"fold":fold,"v5_macro_f1":f5,"v6c_macro_f1":f6,"gain":f6-f5,"vetoes":n,"veto_precision":tp/n if n else np.nan,"corrected_negatives":tp,"false_vetoes":n-tp})
component_metrics=pd.DataFrame(component_rows)

overall_selected=int(crossfit_veto.sum()); overall_correct=int(y_neg[crossfit_veto].sum()); overall_precision=overall_correct/max(1,overall_selected)
# Generator metrics over ALL held-out clean pairs, not only scorable/generated pairs.
gen_total=len(generator_validation); gen_recall=float(generator_validation.generated.mean()) if gen_total else 0.0
coverage_success=float(generator_validation.top1_correct.mean()) if gen_total else 0.0
top1_given_generated=float(generator_validation.loc[generator_validation.generated==1,"top1_correct"].mean()) if int(generator_validation.generated.sum()) else 0.0
TYPE_GEN={}
for typ in TYPES:
    g=generator_validation[generator_validation.slot_type==typ]
    TYPE_GEN[typ]={"n":len(g),"recall":float(g.generated.mean()) if len(g) else np.nan,"coverage_top1":float(g.top1_correct.mean()) if len(g) else np.nan}

finite_by_type={t:[fold_type_threshold[f][t] for f in range(N_FOLDS) if np.isfinite(fold_type_threshold[f][t])] for t in TYPES}
FINAL_TYPE_GATE={t:(float(np.median(v)) if v else np.inf) for t,v in finite_by_type.items()}
print("Generator recall:",gen_recall,"| top1 given generated:",top1_given_generated,"| coverage-adjusted success:",coverage_success)
print("Type generator metrics:",TYPE_GEN)
print("Cross-fitted veto precision:",overall_precision,"selected=",overall_selected,"corrected negatives=",overall_correct)
print("Final type gates:",FINAL_TYPE_GATE)
display(threshold_metrics); display(component_metrics)

Generator recall: 0.07986688851913477 | top1 given generated: 0.4166666666666667 | coverage-adjusted success: 0.033277870216306155
Type generator metrics: {'lexical': {'n': 431, 'recall': 0.0, 'coverage_top1': 0.0}, 'number': {'n': 41, 'recall': 0.1951219512195122, 'coverage_top1': 0.0}, 'polarity': {'n': 129, 'recall': 0.31007751937984496, 'coverage_top1': 0.15503875968992248}}
Cross-fitted veto precision: 0.0 selected= 0 corrected negatives= 0
Final type gates: {'lexical': inf, 'number': inf, 'polarity': inf}


,fold,type,threshold,heldout_selected,heldout_precision,heldout_corrected_negatives,heldout_false_vetoes,fit_selected,fit_precision,fit_recall
0,0,lexical,inf,0,NaN,0,0,0,NaN,0.0
1,0,number,inf,0,NaN,0,0,0,NaN,0.0
2,0,polarity,inf,0,NaN,0,0,0,NaN,0.0
3,1,lexical,inf,0,NaN,0,0,0,NaN,0.0
4,1,number,inf,0,NaN,0,0,0,NaN,0.0
5,1,polarity,inf,0,NaN,0,0,0,NaN,0.0
6,2,lexical,inf,0,NaN,0,0,0,NaN,0.0
7,2,number,inf,0,NaN,0,0,0,NaN,0.0
8,2,polarity,inf,0,NaN,0,0,0,NaN,0.0


,fold,v5_macro_f1,v6c_macro_f1,gain,vetoes,veto_precision,corrected_negatives,false_vetoes
0,0,0.893135,0.893135,0.0,0,NaN,0,0
1,1,0.890470,0.890470,0.0,0,NaN,0,0
2,2,0.899939,0.899939,0.0,0,NaN,0,0


## Repeated pseudo-competition stability diagnostic

The pseudo diagnostic reuses only strict OOF challenger margins and fold-derived thresholds. It never fits on pseudo-query labels. Rows without strict OOF V6C scores simply abstain, making this diagnostic conservative rather than optimistic.

In [12]:
pseudo_rows=[]; idx_all=np.arange(len(train))
for k in range(N_PSEUDO_SPLITS):
    tr_idx,q_idx=train_test_split(idx_all,test_size=.20,random_state=SEED+k,stratify=y)
    ref=train.iloc[tr_idx].reset_index(drop=True); qry=train.iloc[q_idx].reset_index(drop=True)
    base,_,forced=base_v2_for_query(qry,ref,v2_oof_meta[q_idx])
    v4,v4_rules,_,_=apply_transductive_v4(qry,base,forced,rank_oof[q_idx],ref)
    v5,rescue=apply_v5_rescue(v4,v4_rules,v3_oof_blend[q_idx],v2_oof_global[q_idx])
    eligible=(v5==1)&(np.asarray(v4_rules,dtype=object)=="v2_fallback")&(~rescue)
    veto=np.zeros(len(q_idx),dtype=bool)
    for pos,idx in enumerate(q_idx):
        if not eligible[pos]: continue
        fi=int(fold_id[idx])
        for typ in TYPES:
            m=oof_margin[typ][idx]; th=fold_type_threshold[fi][typ]
            if np.isfinite(m) and np.isfinite(th) and m>=th:
                veto[pos]=True; break
    v6=v5.copy(); v6[veto]=0
    f5=f1_score(y[q_idx],v5,average="macro",zero_division=0); f6=f1_score(y[q_idx],v6,average="macro",zero_division=0)
    n=int(veto.sum()); tp=int((y[q_idx][veto]==0).sum())
    pseudo_rows.append({"split":k,"v5_macro_f1":f5,"v6c_macro_f1":f6,"gain":f6-f5,"vetoes":n,"veto_precision":tp/n if n else np.nan})
pseudo_metrics=pd.DataFrame(pseudo_rows)
wins=int((pseudo_metrics.gain>1e-12).sum()); ties=int((pseudo_metrics.gain.abs()<=1e-12).sum()); losses=int((pseudo_metrics.gain<-1e-12).sum())
print(f"Pseudo V6C: wins={wins} ties={ties} losses={losses}/{N_PSEUDO_SPLITS} | mean gain={pseudo_metrics.gain.mean():.6f}")
display(pseudo_metrics.describe())

Pseudo V6C: wins=0 ties=100 losses=0/100 | mean gain=0.000000


,split,v5_macro_f1,v6c_macro_f1,gain,vetoes,veto_precision
count,100.000000,100.000000,100.000000,100.0,100.0,0.0
mean,49.500000,0.921804,0.921804,0.0,0.0,NaN
std,29.011492,0.007675,0.007675,0.0,0.0,NaN
min,0.000000,0.904430,0.904430,0.0,0.0,NaN
25%,24.750000,0.916577,0.916577,0.0,0.0,NaN
50%,49.500000,0.921163,0.921163,0.0,0.0,NaN
75%,74.250000,0.925854,0.925854,0.0,0.0,NaN
max,99.000000,0.940067,0.940067,0.0,0.0,NaN


## Final 2/3 same-slot + same-challenger test consensus and fail-closed acceptance

A test slot qualifies only when at least two fold models choose the **same normalized challenger** for the **same slot**, and their mean challenger advantage clears the type-specific cross-fitted gate. This is the deployment skill V6B learned well, now aligned with inference.

In [13]:
# Candidate-level final consensus.
proposed_veto=np.zeros(len(test),dtype=bool)
consensus_records={}
for idx,slots in test_banks.items():
    best_cons=None
    for slot in slots:
        typ=slot["type"]; gate=FINAL_TYPE_GATE[typ]
        if not np.isfinite(gate): continue
        votes=defaultdict(list)
        for fold in range(N_FOLDS):
            rr=TEST_MODEL_SLOT_RESULTS[fold].get(idx,{}).get(slot["slot_id"])
            if rr is None: continue
            ck=normalize_candidate(rr["challenger"]); votes[ck].append((fold,rr))
        for ck,vals in votes.items():
            if len(vals)<2: continue
            margins=[float(v[1]["margin"]) for v in vals]
            mean_margin=float(np.mean(margins))
            # consensus candidates must clear the common cross-fitted type gate on average.
            if mean_margin<gate: continue
            rec={"slot_id":slot["slot_id"],"type":typ,"original":slot["original"],"challenger":vals[0][1]["challenger"],"votes":len(vals),"mean_margin":mean_margin,"gate":gate,"fold_margins":{str(f):float(r["margin"]) for f,r in vals}}
            if best_cons is None or (rec["mean_margin"],rec["votes"],rec["slot_id"])>(best_cons["mean_margin"],best_cons["votes"],best_cons["slot_id"]): best_cons=rec
    if best_cons is not None:
        proposed_veto[idx]=True; consensus_records[idx]=best_cons

# Primary acceptance gates.
lex_recall=TYPE_GEN["lexical"]["recall"] if np.isfinite(TYPE_GEN["lexical"]["recall"]) else 1.0
num_recall=TYPE_GEN["number"]["recall"] if np.isfinite(TYPE_GEN["number"]["recall"]) else 1.0
pol_recall=TYPE_GEN["polarity"]["recall"] if np.isfinite(TYPE_GEN["polarity"]["recall"]) else 1.0
fold_precision_ok=True
for r in component_metrics.itertuples():
    if r.vetoes>0 and (not np.isfinite(r.veto_precision) or r.veto_precision<MIN_FOLD_PRECISION): fold_precision_ok=False
type_fold_precision_ok=True
for r in threshold_metrics.itertuples():
    if r.heldout_selected>0 and (not np.isfinite(r.heldout_precision) or r.heldout_precision<MIN_FOLD_PRECISION): type_fold_precision_ok=False
validation_accept=(
    gen_recall>=MIN_GENERATOR_RECALL
    and lex_recall>=MIN_LEXICAL_GENERATOR_RECALL
    and num_recall>=MIN_NUMBER_GENERATOR_RECALL
    and pol_recall>=MIN_POLARITY_AVAILABILITY
    and top1_given_generated>=MIN_TOP1_RANKING
    and coverage_success>=MIN_COVERAGE_ADJUSTED_PAIR_SUCCESS
    and overall_precision>=MIN_VETO_PRECISION
    and fold_precision_ok
    and type_fold_precision_ok
    and overall_correct>=MIN_CORRECTED_OOF_NEGATIVES
    and component_metrics.gain.mean()>0
    and wins>=MIN_PSEUDO_WINS
    and losses<=MAX_PSEUDO_LOSSES
    and pseudo_metrics.gain.mean()>0
)
test_veto_count=int(proposed_veto.sum()); test_count_ok=(MIN_TEST_VETOES<=test_veto_count<=MAX_TEST_VETOES)
V6C_ACCEPTED=bool(validation_accept and test_count_ok)
final_v6c=final_v5.copy()
if V6C_ACCEPTED: final_v6c[proposed_veto]=0
else: proposed_veto[:]=False; consensus_records={}

locked_structural=(v5_test.v4_rule.astype(str).to_numpy()!="v2_fallback")
locked_rescue=v5_test.v5_rescue.to_numpy(np.int8).astype(bool)
assert np.array_equal(final_v6c[locked_structural],final_v5[locked_structural])
assert np.array_equal(final_v6c[locked_rescue],final_v5[locked_rescue])
assert not np.any((final_v5==0)&(final_v6c==1)),"V6C attempted forbidden 0->1 change"
assert np.all((~proposed_veto)|test_candidate)
print("V6C validation accepted:",V6C_ACCEPTED)
print("Proposed/active test vetoes:",int(proposed_veto.sum()))
print("Final V5 distribution:",pd.Series(final_v5).value_counts().sort_index().to_dict())
print("Final V6C distribution:",pd.Series(final_v6c).value_counts().sort_index().to_dict())

V6C validation accepted: False
Proposed/active test vetoes: 0
Final V5 distribution: {0: 379, 1: 3224}
Final V6C distribution: {0: 379, 1: 3224}


## Export complete V6C audit bundle

Every final intervention includes the slot, original value, challenger, consensus vote count, margin, and gate. The notebook therefore remains fully auditable even if V6C fails closed.

# submission=sample[["id"]].copy(); submission["label"]=final_v6c.astype(int)
assert submission.id.astype(str).tolist()==test.id.astype(str).tolist()
assert len(submission)==EXPECTED_TEST_ROWS and set(submission.label.unique()).issubset({0,1})

# OOF score audit.
oof_out=pd.DataFrame({"id":train.id,"label":y,"fold":fold_id,"v5_label":oof_v5_label,"v4_rule":oof_v4_rule,"v5_rescue":oof_v5_rescue.astype(int),"v6c_candidate":oof_candidate.astype(int),"crossfit_veto":crossfit_veto.astype(int),"v6c_label":v6c_oof})
for typ in TYPES:
    oof_out[f"margin_{typ}"]=oof_margin[typ]; oof_out[f"slot_{typ}"]=oof_best_slot[typ]; oof_out[f"original_{typ}"]=oof_original[typ]; oof_out[f"challenger_{typ}"]=oof_challenger[typ]
    oof_out[f"threshold_{typ}"]=[fold_type_threshold[int(f)][typ] for f in fold_id]

# Test compact audit.
test_rows=[]
for idx,r in enumerate(test.itertuples()):
    cons=consensus_records.get(idx,{})
    test_rows.append({"id":r.id,"v5_label":int(final_v5[idx]),"v4_rule":str(v5_test.v4_rule.iloc[idx]),"v5_rescue":int(v5_test.v5_rescue.iloc[idx]),"v6c_candidate":int(test_candidate[idx]),"v6c_veto":int(proposed_veto[idx]),"consensus_type":cons.get("type",""),"consensus_slot":cons.get("slot_id",""),"original":cons.get("original",""),"challenger":cons.get("challenger",""),"votes":cons.get("votes",0),"mean_margin":cons.get("mean_margin",np.nan),"gate":cons.get("gate",np.nan),"label":int(final_v6c[idx])})
test_out=pd.DataFrame(test_rows)
changed=test_out.loc[final_v6c!=final_v5].copy().merge(test[["id","title","content"]],on="id",how="left")

config={
    "version":"v6c-article-grounded-counterfactual-slot-challenger-ranker","seed":SEED,"n_folds":N_FOLDS,"epochs":EPOCHS,"base_system":"frozen-v5-champion",
    "model_name":MODEL_NAME,"model_revision":MODEL_REVISION,"model_source":MODEL_SOURCE,"binary_labels":{"0":"UNSUPPORTED","1":"SUPPORTED"},
    "counterfactual_pairs":{"similarity_min":PAIR_SIMILARITY_MIN,"one_edit_only":True,"max_changed_span_tokens":MAX_CHANGED_SPAN_TOKENS,"mined_pairs":len(clean_pairs),"edit_counts":clean_pairs.edit.value_counts().to_dict(),"type_counts":clean_pairs.slot_type.value_counts().to_dict()},
    "candidate_generation":{"shared_evidence":True,"window_words":SLOT_WINDOW_WORDS,"stride":SLOT_WINDOW_STRIDE,"top_windows":SLOT_TOP_WINDOWS,"max_challengers":MAX_CHALLENGERS,"max_lexical_slots":MAX_LEXICAL_SLOTS,"max_numeric_slots":MAX_NUMERIC_SLOTS,"max_polarity_slots":MAX_POLARITY_SLOTS,"full_generator_recall":gen_recall,"type_metrics":TYPE_GEN,"coverage_adjusted_pair_success":coverage_success,"top1_given_generated":top1_given_generated},
    "training":{"control_batch_size":CONTROL_BATCH_SIZE,"grad_accum":GRAD_ACCUM,"lr":LR,"weight_decay":WEIGHT_DECAY,"warmup_ratio":WARMUP_RATIO,"amp":AMP_ENABLED,"data_parallel":GPU_COUNT>=2,"synthetic":False,"negative_row_mil":False,"loss_weights":{"candidate_bce":CANDIDATE_BCE_WEIGHT,"pair_margin":PAIR_LOSS_WEIGHT,"positive_control":POSITIVE_CONTROL_WEIGHT},"pair_margin":PAIR_MARGIN},
    "validation":{"crossfit_veto_precision":overall_precision,"crossfit_selected":overall_selected,"crossfit_corrected_negatives":overall_correct,"component_mean_v5":float(component_metrics.v5_macro_f1.mean()),"component_mean_v6c":float(component_metrics.v6c_macro_f1.mean()),"component_mean_gain":float(component_metrics.gain.mean()),"pseudo_wins":wins,"pseudo_ties":ties,"pseudo_losses":losses,"pseudo_mean_gain":float(pseudo_metrics.gain.mean())},
    "veto":{"type_fold_thresholds":fold_type_threshold,"final_type_gates":FINAL_TYPE_GATE,"same_slot_same_challenger_votes":2,"min_precision":MIN_VETO_PRECISION,"min_fold_precision":MIN_FOLD_PRECISION,"min_test_vetoes":MIN_TEST_VETOES,"max_test_vetoes":MAX_TEST_VETOES},
    "accepted":V6C_ACCEPTED,"test_vetoes":int(proposed_veto.sum()),"dataset_sha256":data_hashes,"frozen_artifact_sha256":{k:v["sha256"] for k,v in FROZEN_ARTIFACTS.items()},
    "runtime":{"torch":torch.__version__,"transformers":transformers.__version__,"gpus":[torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]},
}

exports={
    "submission_v6c.csv":submission,
    "counterfactual_pairs_v6c.csv":clean_pairs,
    "generator_validation_v6c.csv":generator_validation,
    "specialist_fold_metrics_v6c.csv":fold_specialist_diag,
    "training_history_v6c.csv":training_history,
    "oof_slot_scores_v6c.csv":oof_out,
    "crossfit_veto_metrics_v6c.csv":threshold_metrics.merge(component_metrics,on="fold",how="left"),
    "pseudo_competition_v6c.csv":pseudo_metrics,
    "changed_rows_v6c.csv":changed,
    "test_scores_v6c.csv":test_out,
}
for name,df in exports.items(): df.to_csv(OUTPUT_DIR/name,index=False)
def json_safe(x):
    if isinstance(x,dict): return {str(k):json_safe(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)): return [json_safe(v) for v in x]
    if isinstance(x,(np.integer,)): return int(x)
    if isinstance(x,(np.floating,float)):
        v=float(x); return v if np.isfinite(v) else None
    return x
with open(OUTPUT_DIR/"config_v6c.json","w") as f: json.dump(json_safe(config),f,indent=2,allow_nan=False)
zip_path=OUTPUT_DIR/"results_v6c.zip"
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    for name in list(exports)+["config_v6c.json"]: zf.write(OUTPUT_DIR/name,arcname=name)
print("Saved V6C artifacts:")
for name in list(exports)+["config_v6c.json","results_v6c.zip"]: print(" -",OUTPUT_DIR/name)
print("V6C ACCEPTED =",V6C_ACCEPTED,"| changed vs V5 =",int((final_v6c!=final_v5).sum()))
display(submission.head())